# MicCheck: Measuring Recording-Condition Shortcuts in Speech Representations

**Natural paired-condition probe:** the AMI Meeting Corpus provides closely corresponding speech segments recorded with an Individual Headset Microphone (IHM) and a Single Distant Microphone (SDM). MicCheck asks how much lightweight acoustic representations change between close-talk and distant-microphone conditions, whether that change harms useful speech information, which feature families carry the signal, and whether mitigation actually removes recoverable condition information.

### Research questions

1. How accurately can a classifier recover IHM versus SDM recording condition from lightweight acoustic features?
2. How much do closed-set speaker-information and small lexical probes degrade across recording conditions?
3. Which feature families carry recording-condition information after simple amplitude controls?
4. How do source-only normalization and target-aware CORAL adaptation differ?
5. Can matched recordings encourage channel-aligned representations while preserving useful speech information?
6. Does geometric pair alignment correspond to lower representational channel leakage?

**Core distinction:** geometric pair alignment is not necessarily representational invariance.

This notebook can be uploaded directly to Google Colab and run from top to bottom. It streams AMI, limits expensive audio processing with MAX_PAIRS, caches features, and uses no pretrained speech foundation model.


## Hypotheses and scientific scope

- **H1:** Conventional acoustic features encode substantial recording-condition information for held-out speakers.
- **H2:** Closed-set speaker-information and small lexical probes perform better within a recording condition than across conditions.
- **H3:** Source-only preprocessing can reduce some recoverable condition information, with a possible cost to useful information.
- **H4a:** A paired IHM/SDM alignment objective reduces the representation distance between matched recordings while preserving useful speech information.
- **H4b:** Paired alignment may reduce linearly recoverable recording-condition information, but this must be tested directly rather than inferred from pair distance.

IHM and SDM differ in distance, reverberation, room response, noise, placement, gain, and hardware. The contrast therefore measures **close-talk versus distant-microphone recording condition**, not microphone hardware in isolation and not a causal microphone-only effect.

Reducing the distance between matched recordings does not imply that channel information has been removed. H4a and H4b are assessed separately from generated results.


## 1. Self-contained environment setup

The next cell contains all required runtime dependencies directly, so the notebook does not require a cloned repository or a local requirements file. In Google Colab, upload MicCheck_AMI.ipynb and run all cells. The version-reporting cell that follows records the resulting environment.

AMI is accessed through Hugging Face streaming. Audio decoding is requested with decode=False and handled through soundfile where possible; the full corpus is not downloaded.


In [ ]:
%pip install -q "datasets>=5.1,<6" "fsspec>=2024.6" "numpy>=1.26,<3" "pandas>=2.1,<3" "pyarrow>=15" "scipy>=1.12,<2" "librosa>=0.10.2,<1" "soundfile>=0.12,<1" "scikit-learn>=1.4,<2" "matplotlib>=3.8,<4" "torch>=2.2,<3" "tqdm>=4.66,<5" "psutil>=5.9,<8"


In [ ]:
import gc, hashlib, io, json, os, random, re, shutil, sys, warnings
from collections import Counter, defaultdict
from pathlib import Path

import datasets, fsspec, librosa, matplotlib, matplotlib.pyplot as plt
import numpy as np, pandas as pd, scipy, soundfile as sf, sklearn, torch
from datasets import Audio as HFAudio, load_dataset
from IPython.display import Audio, Markdown, display
from scipy import ndimage, signal
from sklearn.base import clone
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, confusion_matrix,
                             ConfusionMatrixDisplay, f1_score, roc_auc_score)
from sklearn.model_selection import GroupKFold, GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler, StandardScaler
from sklearn.svm import SVC
from tqdm.auto import tqdm

print("Python", sys.version.split()[0])
for package in (datasets, np, pd, scipy, librosa, sklearn, matplotlib, torch):
    print(f"{package.__name__:12s} {package.__version__}")


## 2. Configuration and reproducibility

Protocol version 2.0 separates complete metadata discovery from bounded audio processing, adds balanced sampling, feature and amplitude ablations, grouped robustness evaluation, clustered uncertainty, and standardized frozen probes. The committed repository results are a protocol-v1 snapshot; all v2 values require a fresh notebook run.

MAX_PAIRS limits expensive audio processing, not metadata discovery. All random operations use SEED, and cache signatures include protocol and feature-extraction settings.


In [ ]:
PROTOCOL_VERSION = "2.0"
SEED = 42
RUN_MODE = "standard"  # "smoke", "standard", or "extended"
MAX_PAIRS = {"smoke": 200, "standard": 2000, "extended": 5000}[RUN_MODE]
AMI_SPLIT = "validation"
METADATA_SCAN_LIMIT = None  # None scans all metadata rows; MAX_PAIRS applies only after pairing
MIN_DURATION, MAX_DURATION = 0.5, 10.0
FORCE_RECOMPUTE_FEATURES = False

MIN_WORD_COUNT, TOP_K_WORDS = 15, 10
LEXICAL_ASYMMETRY_THRESHOLD = 0.10
LAMBDA_VALUES = [0.0, 0.01, 0.1, 0.5, 1.0]
N_BOOTSTRAPS = 100 if RUN_MODE == "smoke" else 500
COUNTERFACTUAL_TRAIN_PAIRS = {"smoke": 30, "standard": 120, "extended": 250}[RUN_MODE]
WAVEFORM_REPRESENTATIONS = ("raw", "peak_norm", "rms_norm")
TARGET_RMS = 0.05

PROJECT_DIR = Path.cwd()
CACHE_DIR = PROJECT_DIR / "cache"
RESULTS_DIR = PROJECT_DIR / "results"
FIGURE_DIR = RESULTS_DIR / "figures"
for directory in (CACHE_DIR, RESULTS_DIR, FIGURE_DIR):
    directory.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
try:
    torch.use_deterministic_algorithms(True, warn_only=True)
except Exception:
    pass
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

try:
    import psutil
    ram_gb = psutil.virtual_memory().total / 2**30
except Exception:
    ram_gb = float("nan")
disk_gb = shutil.disk_usage(PROJECT_DIR).free / 2**30
print(f"Protocol: {PROTOCOL_VERSION}\nRuntime device: {device}\nRAM: {ram_gb:.1f} GiB\nAvailable disk: {disk_gb:.1f} GiB")
print(f"Mode: {RUN_MODE}; audio-processing budget: {MAX_PAIRS:,} pairs")


## 3. Stream AMI and inspect the live schema

Metadata discovery and audio access are separate. The metadata pass removes the audio column and scans the selected AMI split before any subset is chosen. A later pass revisits the stream and decodes only the deterministically sampled audio IDs. Pairing never assumes common row order.


In [ ]:
DATASET_NAME = "edinburghcstr/ami"
REQUIRED_COLUMNS = {"meeting_id","audio_id","text","audio","begin_time","end_time","microphone_id","speaker_id"}

def open_ami(config, include_audio=False):
    ds = load_dataset(DATASET_NAME, config, split=AMI_SPLIT, streaming=True)
    missing = REQUIRED_COLUMNS - set(ds.column_names)
    if missing: raise RuntimeError(f"AMI/{config} is missing required columns: {sorted(missing)}")
    try:
        ds = ds.cast_column("audio", HFAudio(sampling_rate=16000, decode=False))
    except Exception as cast_exc:
        try:
            ds = ds.decode(False)
        except Exception as decode_exc:
            raise RuntimeError(
                "Unable to disable Hugging Face audio decoding; update `datasets` or install a compatible version. "
                f"cast error={cast_exc}; decode error={decode_exc}"
            ) from decode_exc
    return ds if include_audio else ds.remove_columns(["audio"])

for config in ("ihm", "sdm"):
    sample = next(iter(open_ami(config, include_audio=False)))
    print(config.upper(), {k: (type(v).__name__, str(v)[:70]) for k,v in sample.items()})


## 4. Discover all valid metadata pairs, then sample for diversity

Rows are matched by meeting, speaker, normalized transcript, and begin/end times within 20 ms. Phase A scans metadata and builds every valid IHM/SDM pair available under METADATA_SCAN_LIMIT. Phase B deterministically shuffles within meeting–speaker groups and samples round-robin across groups.

This design prevents MAX_PAIRS from selecting only the earliest stream rows: MAX_PAIRS limits expensive audio processing, not metadata discovery.


In [ ]:
def normalize_text(text):
    return re.sub(r"\s+", " ", re.sub(r"[^A-Z0-9']+", " ", str(text).upper())).strip()

def eligible_metadata(row):
    try:
        duration = float(row["end_time"]) - float(row["begin_time"])
        return (MIN_DURATION <= duration <= MAX_DURATION and bool(normalize_text(row["text"]))
                and bool(row["meeting_id"]) and bool(row["speaker_id"]) and bool(row["audio_id"]))
    except Exception:
        return False

def coarse_key(row):
    return (str(row["meeting_id"]), str(row["speaker_id"]), normalize_text(row["text"]))

def exactish_time(row):
    return (round(float(row["begin_time"]) * 100), round(float(row["end_time"]) * 100))

def within_metadata_limit(index):
    return METADATA_SCAN_LIMIT is None or index < METADATA_SCAN_LIMIT

exclusions = []
ihm_index = defaultdict(list)
for i, row in enumerate(tqdm(open_ami("ihm"), desc="IHM metadata discovery")):
    if not within_metadata_limit(i):
        break
    if eligible_metadata(row):
        ihm_index[coarse_key(row)].append(dict(row))
    else:
        exclusions.append({"config": "ihm", "audio_id": row.get("audio_id"), "reason": "metadata_filter"})

used_ihm, discovered_pairs = set(), []
for i, sdm in enumerate(tqdm(open_ami("sdm"), desc="SDM metadata pairing")):
    if not within_metadata_limit(i):
        break
    if not eligible_metadata(sdm):
        exclusions.append({"config": "sdm", "audio_id": sdm.get("audio_id"), "reason": "metadata_filter"})
        continue
    sb, se = exactish_time(sdm)
    candidates = []
    for ihm in ihm_index.get(coarse_key(sdm), []):
        if ihm["audio_id"] in used_ihm:
            continue
        ib, ie = exactish_time(ihm)
        if abs(sb - ib) <= 2 and abs(se - ie) <= 2:
            candidates.append((abs(sb - ib) + abs(se - ie), str(ihm["audio_id"]), ihm))
    if not candidates:
        continue
    _, _, ihm = min(candidates, key=lambda item: (item[0], item[1]))
    used_ihm.add(ihm["audio_id"])
    signature = "|".join(map(str, (*coarse_key(ihm), *exactish_time(ihm), ihm["audio_id"], sdm["audio_id"])))
    discovered_pairs.append({
        "pair_id": hashlib.sha1(signature.encode()).hexdigest()[:16],
        "meeting_id": ihm["meeting_id"], "speaker_id": ihm["speaker_id"], "text": ihm["text"],
        "normalized_text": normalize_text(ihm["text"]), "begin_time": float(ihm["begin_time"]),
        "end_time": float(ihm["end_time"]), "duration": float(ihm["end_time"]) - float(ihm["begin_time"]),
        "ihm_audio_id": ihm["audio_id"], "sdm_audio_id": sdm["audio_id"],
        "ihm_microphone_id": ihm["microphone_id"], "sdm_microphone_id": sdm["microphone_id"],
        "time_error_ms": 10 * (abs(sb - exactish_time(ihm)[0]) + abs(se - exactish_time(ihm)[1]))})

all_pairs_df = pd.DataFrame(discovered_pairs)
if all_pairs_df.empty:
    raise RuntimeError("No AMI pairs found. Inspect the schema samples and Hugging Face status.")
if all_pairs_df.pair_id.duplicated().any():
    raise AssertionError("Duplicate pair IDs after metadata matching")
all_discovered_pairs = len(all_pairs_df)

def balanced_pair_sample(frame, limit, seed):
    if len(frame) <= limit:
        return frame.sort_values(["meeting_id", "speaker_id", "pair_id"]).reset_index(drop=True)
    generator = np.random.default_rng(seed)
    grouped = {}
    for group_index, (key, group) in enumerate(frame.groupby(["meeting_id", "speaker_id"], sort=True)):
        order = generator.permutation(len(group))
        grouped[key] = group.iloc[order].to_dict("records")
    keys = sorted(grouped)
    generator.shuffle(keys)
    selected = []
    while len(selected) < limit and any(grouped[key] for key in keys):
        for key in keys:
            if grouped[key] and len(selected) < limit:
                selected.append(grouped[key].pop())
    return pd.DataFrame(selected).reset_index(drop=True)

pair_df = balanced_pair_sample(all_pairs_df, MAX_PAIRS, SEED)
assert pair_df.pair_id.nunique() == len(pair_df)
assert set(pair_df.pair_id).issubset(set(all_pairs_df.pair_id))
pair_df.to_csv(RESULTS_DIR / "paired_manifest.csv", index=False)
excluded_df = pd.DataFrame(exclusions)
excluded_df.to_csv(RESULTS_DIR / "excluded_samples.csv", index=False)
print(f"Discovered {all_discovered_pairs:,} valid metadata pairs; sampled {len(pair_df):,} for audio processing.")
print(f"Sample diversity: {pair_df.speaker_id.nunique()} speakers across {pair_df.meeting_id.nunique()} meetings.")
print(f"Maximum accepted timestamp error: {pair_df.time_error_ms.max():.0f} ms")


## 5. Dataset audit

The audit distinguishes all discovered metadata pairs from the balanced audio-processing subset. Speaker and meeting distributions are reported explicitly because concentration can create optimistic recording-condition estimates. The full sampled manifest is exported; metadata discovery itself is not capped by MAX_PAIRS.


In [ ]:
def distribution_stats(counts, prefix):
    counts = pd.Series(counts, dtype=float)
    return {
        f"{prefix}_min": counts.min() if len(counts) else np.nan,
        f"{prefix}_median": counts.median() if len(counts) else np.nan,
        f"{prefix}_max": counts.max() if len(counts) else np.nan,
    }

speaker_counts = pair_df.groupby("speaker_id").pair_id.nunique()
meeting_counts = pair_df.groupby("meeting_id").pair_id.nunique()
summary = pd.DataFrame([{
    "protocol_version": PROTOCOL_VERSION,
    "run_mode": RUN_MODE,
    "all_discovered_pairs": all_discovered_pairs,
    "sampled_pairs": len(pair_df),
    "speakers": pair_df.speaker_id.nunique(),
    "meetings": pair_df.meeting_id.nunique(),
    **distribution_stats(speaker_counts, "pairs_per_speaker"),
    **distribution_stats(meeting_counts, "pairs_per_meeting"),
    "median_duration_s": pair_df.duration.median(),
    "total_paired_audio_hours": 2 * pair_df.duration.sum() / 3600,
    "unique_transcripts": pair_df.normalized_text.nunique(),
    "metadata_exclusions": len(excluded_df),
}])
summary.to_csv(RESULTS_DIR / "dataset_summary.csv", index=False)
display(summary)
print("IHM microphones:", pair_df.ihm_microphone_id.value_counts().to_dict())
print("SDM microphones:", pair_df.sdm_microphone_id.value_counts().to_dict())
display(pair_df.sample(min(8, len(pair_df)), random_state=SEED)[["speaker_id", "meeting_id", "text", "duration"]])

fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))
axes[0].hist(pair_df.duration, bins=30)
axes[0].set(xlabel="Duration (s)", ylabel="Pairs", title="Utterance duration")
speaker_counts.head(20).sort_values().plot.barh(ax=axes[1])
axes[1].set(xlabel="Pairs", title="Pairs per speaker")
meeting_counts.head(20).sort_values().plot.barh(ax=axes[2])
axes[2].set(xlabel="Pairs", title="Pairs per meeting")
fig.suptitle("Balanced AMI paired-subset composition")
plt.tight_layout()
fig.savefig(FIGURE_DIR / "dataset_audit.png", dpi=200)
plt.show()


## 6. Defensive selective audio access

`decode=False` avoids changing `datasets`/`torchcodec` APIs. The helper accepts encoded bytes, local paths, URLs, `hf://`
paths, or a newer decoder object. Failed records are collected. Only requested IDs are decoded and each waveform is
converted to mono 16 kHz float32.


In [ ]:
def decode_audio_object(obj, target_sr=16000):
    if hasattr(obj,"get_all_samples"):
        samples=obj.get_all_samples(); array=np.asarray(samples.data).squeeze(); sr=int(samples.sample_rate)
    elif isinstance(obj,dict):
        if obj.get("array") is not None:
            array=np.asarray(obj["array"]); sr=int(obj.get("sampling_rate",target_sr))
        elif obj.get("bytes") is not None:
            array,sr=sf.read(io.BytesIO(obj["bytes"]),dtype="float32",always_2d=False)
        elif obj.get("path"):
            with fsspec.open(obj["path"],"rb") as handle: array,sr=sf.read(handle,dtype="float32",always_2d=False)
        else: raise ValueError("Audio dictionary has no array, bytes, or path")
    elif isinstance(obj,(str,Path)):
        with fsspec.open(str(obj),"rb") as handle: array,sr=sf.read(handle,dtype="float32",always_2d=False)
    else: raise TypeError(f"Unsupported audio object: {type(obj)}")
    array=np.asarray(array,dtype=np.float32)
    if array.ndim>1: array=array.mean(axis=0 if array.shape[0]<array.shape[1] else 1)
    if sr!=target_sr:
        divisor=np.gcd(sr,target_sr); array=signal.resample_poly(array,target_sr//divisor,sr//divisor).astype(np.float32); sr=target_sr
    if not np.isfinite(array).all() or len(array)<target_sr*MIN_DURATION/2: raise ValueError("Malformed or too-short waveform")
    return array,sr

audio_failures=[]
def stream_selected(config, audio_id_to_pair):
    remaining=set(audio_id_to_pair)
    stream=open_ami(config,include_audio=True)
    for row in tqdm(stream,desc=f"{config.upper()} selective audio"):
        audio_id=row.get("audio_id")
        if audio_id not in remaining: continue
        try:
            audio,sr=decode_audio_object(row["audio"])
            yield audio_id,audio,sr
        except Exception as exc:
            audio_failures.append({"config":config,"audio_id":audio_id,"reason":f"{type(exc).__name__}: {exc}"})
        remaining.remove(audio_id)
        if not remaining: break
    for audio_id in sorted(remaining): audio_failures.append({"config":config,"audio_id":audio_id,"reason":"not_found_in_stream"})

demo_pairs=pair_df.sample(min(3,len(pair_df)),random_state=SEED)
demo_audio={}
for config,column in (("ihm","ihm_audio_id"),("sdm","sdm_audio_id")):
    wanted=dict(zip(demo_pairs[column],demo_pairs.pair_id))
    for audio_id,audio,sr in stream_selected(config,wanted): demo_audio[(wanted[audio_id],config)]=(audio,sr)
print("Decoded demo recordings:",len(demo_audio),"of",2*len(demo_pairs))


## 7. Listen to paired examples and inspect signal differences

These widgets make the recording-condition contrast tangible. The plots show waveforms, log-mel spectrograms, average
spectra, and the SDM−IHM log-spectral difference. Amplitude and timing differences are left visible rather than normalized
away.


In [ ]:
for row in demo_pairs.itertuples():
    print(f"{row.speaker_id} | {row.meeting_id} | {row.duration:.2f}s\n{row.text}\nIHM")
    if (row.pair_id,"ihm") in demo_audio: display(Audio(demo_audio[(row.pair_id,"ihm")][0],rate=16000))
    print("SDM")
    if (row.pair_id,"sdm") in demo_audio: display(Audio(demo_audio[(row.pair_id,"sdm")][0],rate=16000))


In [ ]:
def logmel(y,sr=16000):
    mel=librosa.feature.melspectrogram(y=y,sr=sr,n_fft=512,hop_length=160,n_mels=64,power=2)
    return librosa.power_to_db(mel,ref=np.max)

for row in demo_pairs.head(2).itertuples():
    if any((row.pair_id,c) not in demo_audio for c in ("ihm","sdm")): continue
    h,_=demo_audio[(row.pair_id,"ihm")]; d,_=demo_audio[(row.pair_id,"sdm")]
    fig,axes=plt.subplots(2,2,figsize=(13,7))
    for ax,y,label in ((axes[0,0],h,"IHM"),(axes[0,1],d,"SDM")):
        ax.plot(np.arange(len(y))/16000,y,lw=.5); ax.set(title=f"{label} waveform",xlabel="Time (s)",ylabel="Amplitude")
    for ax,y,label in ((axes[1,0],h,"IHM"),(axes[1,1],d,"SDM")):
        L=logmel(y); extent=[0,len(y)/16000,0,8000]; im=ax.imshow(L,origin="lower",aspect="auto",extent=extent,cmap="magma")
        ax.set(title=f"{label} log-mel",xlabel="Time (s)",ylabel="Frequency (Hz)"); fig.colorbar(im,ax=ax,label="dB")
    fig.suptitle(f"Matched event: {row.meeting_id}, {row.speaker_id}"); plt.tight_layout();
    fig.savefig(FIGURE_DIR/f"pair_{row.pair_id}_wave_mel.png",dpi=200); plt.show()
    f,ph=signal.welch(h,16000,nperseg=1024); _,pd_=signal.welch(d,16000,nperseg=1024)
    hd=10*np.log10(ph+1e-12); dd=10*np.log10(pd_+1e-12)
    fig,axes=plt.subplots(1,2,figsize=(12,4)); axes[0].plot(f,hd,label="IHM"); axes[0].plot(f,dd,label="SDM")
    axes[0].set(xlabel="Frequency (Hz)",ylabel="PSD (dB/Hz)",title="Average spectra"); axes[0].legend()
    axes[1].plot(f,dd-hd); axes[1].axhline(0,color="black",lw=.7); axes[1].set(xlabel="Frequency (Hz)",ylabel="SDM − IHM (dB)",title="Paired spectral difference")
    plt.tight_layout(); fig.savefig(FIGURE_DIR/f"pair_{row.pair_id}_spectra.png",dpi=200); plt.show()


## 8. Extract amplitude-controlled, interpretable acoustic features

Each recording becomes a fixed vector containing MFCC, delta, and delta-delta summaries; spectral shape, RMS energy, and zero-crossing summaries; pitch and voiced fraction; and duration. The extractor creates three waveform conditions without overwriting the input: raw, peak normalized, and RMS normalized.

MFCC-CMN and MFCC-CMVN alter only the cepstral frame sequence. Spectral statistics, RMS, ZCR, pitch, voiced fraction, and duration remain unchanged. Therefore the notebook never treats MFCC-CMVN as global normalization of the combined representation.

The cache signature includes protocol version, waveform-normalization settings, the sampled pair digest, and feature version so stale v1 features cannot be mistaken for v2 results.


In [ ]:
N_MFCC, N_FFT, HOP = 13, 512, 160
FEATURE_EXTRACTION_VERSION = 4
STAT_FEATURES = ["centroid", "bandwidth", "rolloff", "flatness", "rms", "zcr"] + [f"contrast_{i}" for i in range(7)]

def peak_normalize(y, eps=1e-8):
    output = np.asarray(y, dtype=np.float32).copy()
    peak = float(np.max(np.abs(output), initial=0.0))
    return output / peak if peak > eps else output

def rms_normalize(y, target_rms=TARGET_RMS, eps=1e-8):
    output = np.asarray(y, dtype=np.float32).copy()
    current_rms = float(np.sqrt(np.mean(output ** 2))) if output.size else 0.0
    return output * (target_rms / current_rms) if current_rms > eps else output

def waveform_variants(y):
    original = np.asarray(y, dtype=np.float32)
    return {
        "raw": original.copy(),
        "peak_norm": peak_normalize(original),
        "rms_norm": rms_normalize(original),
    }

def summarize_frames(x):
    x = np.atleast_2d(np.asarray(x, float))
    return np.r_[np.nanmean(x, axis=1), np.nanstd(x, axis=1)]

def acoustic_vector(y, sr=16000, mfcc_mode="raw"):
    if len(y) < N_FFT:
        y = np.pad(y, (0, N_FFT - len(y)))
    spectrum = np.abs(librosa.stft(y, n_fft=N_FFT, hop_length=HOP, win_length=400))
    power = spectrum ** 2
    log_mel = librosa.power_to_db(librosa.feature.melspectrogram(S=power, sr=sr, n_mels=40) + 1e-10)
    mfcc = librosa.feature.mfcc(S=log_mel, n_mfcc=N_MFCC)
    if mfcc_mode == "mfcc_cmn":
        mfcc = mfcc - mfcc.mean(axis=1, keepdims=True)
    elif mfcc_mode == "mfcc_cmvn":
        mfcc = (mfcc - mfcc.mean(axis=1, keepdims=True)) / (mfcc.std(axis=1, keepdims=True) + 1e-8)
    elif mfcc_mode != "raw":
        raise ValueError(f"Unknown MFCC mode: {mfcc_mode}")

    width = min(9, mfcc.shape[1] if mfcc.shape[1] % 2 else mfcc.shape[1] - 1)
    if width >= 3:
        delta = librosa.feature.delta(mfcc, width=width, mode="nearest")
        delta2 = librosa.feature.delta(mfcc, width=width, order=2, mode="nearest")
    else:
        delta = np.zeros_like(mfcc)
        delta2 = np.zeros_like(mfcc)

    spectral = {
        "centroid": librosa.feature.spectral_centroid(S=spectrum, sr=sr),
        "bandwidth": librosa.feature.spectral_bandwidth(S=spectrum, sr=sr),
        "rolloff": librosa.feature.spectral_rolloff(S=spectrum, sr=sr),
        "flatness": librosa.feature.spectral_flatness(S=spectrum),
        "rms": librosa.feature.rms(S=spectrum, frame_length=N_FFT),
        "zcr": librosa.feature.zero_crossing_rate(y, frame_length=N_FFT, hop_length=HOP),
        "contrast": librosa.feature.spectral_contrast(S=spectrum, sr=sr),
    }
    rms_frames = spectral["rms"].ravel()
    threshold = max(1e-5, np.percentile(rms_frames, 30) * 1.5)
    try:
        f0 = librosa.yin(y, fmin=60, fmax=450, sr=sr, frame_length=1024, hop_length=HOP)
        n = min(len(f0), len(rms_frames))
        voiced = rms_frames[:n] > threshold
        f0 = np.asarray(f0[:n])
        f0[~voiced] = np.nan
        finite = f0[np.isfinite(f0)]
        pitch = [np.median(finite), np.mean(finite), np.std(finite), np.mean(voiced)] if finite.size else [0, 0, 0, 0]
    except Exception:
        pitch = [0, 0, 0, 0]
    spectral_matrix = np.vstack(
        [spectral[key] for key in ("centroid", "bandwidth", "rolloff", "flatness", "rms", "zcr")]
        + [spectral["contrast"]]
    )
    tail = np.r_[summarize_frames(spectral_matrix), pitch, len(y) / sr]
    return np.asarray(np.r_[summarize_frames(np.vstack([mfcc, delta, delta2])), tail], dtype=np.float32)

def acoustic_variants(y, sr=16000):
    waves = waveform_variants(y)
    return {
        "raw": acoustic_vector(waves["raw"], sr, "raw"),
        "mfcc_cmn": acoustic_vector(waves["raw"], sr, "mfcc_cmn"),
        "mfcc_cmvn": acoustic_vector(waves["raw"], sr, "mfcc_cmvn"),
        "peak_norm": acoustic_vector(waves["peak_norm"], sr, "raw"),
        "rms_norm": acoustic_vector(waves["rms_norm"], sr, "raw"),
    }

feature_names = []
for stat in ("mean", "std"):
    for group in ("mfcc", "delta", "delta2"):
        feature_names += [f"{group}_{i:02d}_{stat}" for i in range(N_MFCC)]
for stat in ("mean", "std"):
    feature_names += [f"{name}_{stat}" for name in STAT_FEATURES]
feature_names += ["f0_median", "f0_mean", "f0_std", "voiced_fraction", "duration_feature"]

mfcc_only = [name for name in feature_names if name.startswith("mfcc_")]
cepstral = [name for name in feature_names if name.startswith(("mfcc_", "delta_", "delta2_"))]
spectral_only = [name for name in feature_names if name.startswith(("centroid_", "bandwidth_", "rolloff_", "flatness_", "contrast_", "zcr_"))]
energy_only = [name for name in feature_names if name.startswith("rms_")]
pitch_voicing = [name for name in feature_names if name.startswith("f0_") or name == "voiced_fraction"]
FEATURE_GROUPS = {
    "All features": feature_names,
    "MFCC only": mfcc_only,
    "MFCC + delta + delta-delta": cepstral,
    "Spectral only": spectral_only,
    "Energy only": energy_only,
    "Pitch/voicing only": pitch_voicing,
    "Everything except energy": [name for name in feature_names if name not in energy_only],
    "Everything except duration": [name for name in feature_names if name != "duration_feature"],
    "Everything except energy and duration": [name for name in feature_names if name not in energy_only and name != "duration_feature"],
}
assert all(columns for columns in FEATURE_GROUPS.values())
assert len(feature_names) == len(acoustic_variants(np.zeros(16000))["raw"])

pair_digest = hashlib.sha1("|".join(sorted(pair_df.pair_id)).encode()).hexdigest()[:12]
cache_settings = {
    "protocol_version": PROTOCOL_VERSION,
    "dataset": DATASET_NAME,
    "split": AMI_SPLIT,
    "run_mode": RUN_MODE,
    "pair_digest": pair_digest,
    "min_duration": MIN_DURATION,
    "max_duration": MAX_DURATION,
    "feature_version": FEATURE_EXTRACTION_VERSION,
    "waveform_representations": WAVEFORM_REPRESENTATIONS,
    "target_rms": TARGET_RMS,
}
cache_signature = hashlib.sha1(json.dumps(cache_settings, sort_keys=True).encode()).hexdigest()[:12]
feature_cache = CACHE_DIR / f"ami_features_{RUN_MODE}_v{PROTOCOL_VERSION}_{cache_signature}.parquet"

if feature_cache.exists() and not FORCE_RECOMPUTE_FEATURES:
    feature_df = pd.read_parquet(feature_cache)
    print("Loaded", feature_cache)
else:
    feature_rows = []
    meta_by_id = {}
    for row in pair_df.itertuples():
        meta_by_id[row.ihm_audio_id] = (row.pair_id, "IHM", row)
        meta_by_id[row.sdm_audio_id] = (row.pair_id, "SDM", row)
    for config, column in (("ihm", "ihm_audio_id"), ("sdm", "sdm_audio_id")):
        wanted = {getattr(row, column): row.pair_id for row in pair_df.itertuples()}
        for audio_id, y, sr in stream_selected(config, wanted):
            pair_id, channel, row = meta_by_id[audio_id]
            try:
                if abs(len(y) / sr - row.duration) > 0.35:
                    raise ValueError(f"decoded duration {len(y) / sr:.3f}s differs from metadata {row.duration:.3f}s")
                for representation, vector in acoustic_variants(y, sr).items():
                    record = {
                        "protocol_version": PROTOCOL_VERSION,
                        "pair_id": pair_id,
                        "channel": channel,
                        "representation": representation,
                        "speaker_id": row.speaker_id,
                        "meeting_id": row.meeting_id,
                        "text": row.text,
                        "normalized_text": row.normalized_text,
                        "audio_id": audio_id,
                    }
                    record.update(dict(zip(feature_names, vector)))
                    feature_rows.append(record)
            except Exception as exc:
                audio_failures.append({"config": config, "audio_id": audio_id, "reason": f"feature_error: {type(exc).__name__}: {exc}"})
            del y
            gc.collect()
    if not feature_rows:
        pd.DataFrame(audio_failures).to_csv(RESULTS_DIR / "audio_failures.csv", index=False)
        raise RuntimeError("No audio features were extracted. Inspect audio_failures.csv and fix decoding or network access.")
    feature_df = pd.DataFrame(feature_rows)
    complete = feature_df.groupby(["pair_id", "representation"]).channel.nunique().unstack(fill_value=0)
    required = ["raw", "mfcc_cmn", "mfcc_cmvn", "peak_norm", "rms_norm"]
    if not set(required).issubset(complete.columns):
        raise RuntimeError("One or more required v2 feature representations are missing.")
    valid_pairs = set(complete.index[complete[required].eq(2).all(axis=1)])
    feature_df = feature_df[feature_df.pair_id.isin(valid_pairs)].copy()
    if feature_df.empty:
        raise RuntimeError("No pair has both channels for every required v2 representation.")
    feature_df.to_parquet(feature_cache, index=False)

feature_df.to_parquet(RESULTS_DIR / "features.parquet", index=False)
pd.DataFrame(audio_failures, columns=["config", "audio_id", "reason"]).to_csv(RESULTS_DIR / "audio_failures.csv", index=False)
print(f"Feature rows: {len(feature_df):,}; complete pairs: {feature_df.pair_id.nunique():,}; dimension: {len(feature_names)}")
print("Representations:", sorted(feature_df.representation.unique()))
print("Audio exclusions:", len(audio_failures), Counter(item["reason"].split(":")[0] for item in audio_failures))


## 9. Leakage-safe partitions and paired feature movement

The primary recording-condition partition groups by speaker. Every pair and both channels remain in exactly one partition. The closed-set speaker-information probe later uses the same identities but disjoint pairs.

For descriptive pair-distance plots, the scaler is fit only on primary-training pairs and applied to held-out pairs. Matched pairs share speech content; unmatched pairs change both content and recording condition, so distance remains descriptive rather than causal.


In [ ]:
pair_meta = pair_df[pair_df.pair_id.isin(feature_df.pair_id.unique())].reset_index(drop=True)
if pair_meta.speaker_id.nunique() < 6:
    raise RuntimeError("Fewer than six speakers remain after feature extraction; grouped evaluation is unstable.")

outer = GroupShuffleSplit(n_splits=1, train_size=0.70, random_state=SEED)
train_i, rest_i = next(outer.split(pair_meta, groups=pair_meta.speaker_id))
rest = pair_meta.iloc[rest_i].reset_index(drop=True)
inner = GroupShuffleSplit(n_splits=1, train_size=0.50, random_state=SEED + 1)
val_j, test_j = next(inner.split(rest, groups=rest.speaker_id))
split_map = {pair_id: "train" for pair_id in pair_meta.iloc[train_i].pair_id}
split_map.update({pair_id: "validation" for pair_id in rest.iloc[val_j].pair_id})
split_map.update({pair_id: "test" for pair_id in rest.iloc[test_j].pair_id})
pair_meta["split"] = pair_meta.pair_id.map(split_map)
feature_df["split"] = feature_df.pair_id.map(split_map)

train_s = set(pair_meta.loc[pair_meta.split == "train", "speaker_id"])
val_s = set(pair_meta.loc[pair_meta.split == "validation", "speaker_id"])
test_s = set(pair_meta.loc[pair_meta.split == "test", "speaker_id"])
assert train_s.isdisjoint(val_s | test_s) and val_s.isdisjoint(test_s)
assert feature_df.groupby("pair_id").split.nunique().max() == 1
assert feature_df.groupby(["pair_id", "representation"]).channel.nunique().min() == 2

raw = feature_df[feature_df.representation == "raw"].copy()
headset = raw[raw.channel == "IHM"].set_index("pair_id")
distant = raw[raw.channel == "SDM"].set_index("pair_id")
common = headset.index.intersection(distant.index)
H = headset.loc[common, feature_names].to_numpy()
D = distant.loc[common, feature_names].to_numpy()
train_mask = common.isin(pair_meta.loc[pair_meta.split == "train", "pair_id"])
scaler_distance = StandardScaler().fit(np.vstack([H[train_mask], D[train_mask]]))
Hs, Ds = scaler_distance.transform(H), scaler_distance.transform(D)
generator = np.random.default_rng(SEED)
permutation = generator.permutation(len(common))
mfcc_idx = [i for i, name in enumerate(feature_names) if name.startswith(("mfcc_", "delta_", "delta2_"))]
spectral_idx = [i for i in range(len(feature_names)) if i not in mfcc_idx]

def rowwise_cosine_distance(a, b):
    return 1 - np.sum(a * b, axis=1) / (np.linalg.norm(a, axis=1) * np.linalg.norm(b, axis=1) + 1e-9)

distances = pd.DataFrame({
    "matched_euclidean": np.linalg.norm(Hs - Ds, axis=1),
    "unmatched_euclidean": np.linalg.norm(Hs - Ds[permutation], axis=1),
    "matched_cosine": rowwise_cosine_distance(Hs, Ds),
    "unmatched_cosine": rowwise_cosine_distance(Hs, Ds[permutation]),
    "matched_mfcc": np.linalg.norm(Hs[:, mfcc_idx] - Ds[:, mfcc_idx], axis=1),
    "matched_non_mfcc": np.linalg.norm(Hs[:, spectral_idx] - Ds[:, spectral_idx], axis=1),
})
display(distances.describe().loc[["mean", "50%", "std"]])
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
axes[0].hist(distances.matched_euclidean, bins=35, alpha=0.65, label="matched")
axes[0].hist(distances.unmatched_euclidean, bins=35, alpha=0.65, label="unmatched")
axes[0].set(xlabel="Train-standardized Euclidean distance", ylabel="Pairs", title="Cross-condition feature distance")
axes[0].legend()
axes[1].hist(distances.matched_cosine, bins=35, alpha=0.65, label="matched")
axes[1].hist(distances.unmatched_cosine, bins=35, alpha=0.65, label="unmatched")
axes[1].set(xlabel="Cosine distance", ylabel="Pairs", title="Angular feature distance")
axes[1].legend()
plt.tight_layout()
fig.savefig(FIGURE_DIR / "matched_unmatched_distances.png", dpi=200)
plt.show()


## 10. Speaker-held-out and meeting-held-out protocols

The primary train/validation/test split is speaker-held-out. A separate GroupKFold robustness analysis holds out complete meetings. These protocols are reported separately and never averaged: speaker grouping tests transfer to unseen voices, while meeting grouping tests whether recording-condition recovery depends on session or room characteristics.

All fold-specific preprocessing and classifiers are fit only on their fold's training rows. Pair IDs, and therefore both channels of every speech event, remain within a single fold.


In [ ]:
def grouped_microphone_evaluation(frame, columns, group_column, protocol_label, max_splits=5):
    pair_groups = frame[["pair_id", group_column]].drop_duplicates()
    if pair_groups.groupby("pair_id")[group_column].nunique().max() != 1:
        raise AssertionError(f"A pair maps to multiple {group_column} values")
    n_groups = pair_groups[group_column].nunique()
    if n_groups < 2:
        return pd.DataFrame()
    splitter = GroupKFold(n_splits=min(max_splits, n_groups))
    rows = []
    for fold, (train_index, test_index) in enumerate(
        splitter.split(pair_groups, groups=pair_groups[group_column]), start=1
    ):
        train_pairs = set(pair_groups.iloc[train_index].pair_id)
        test_pairs = set(pair_groups.iloc[test_index].pair_id)
        assert train_pairs.isdisjoint(test_pairs)
        train_rows = frame.pair_id.isin(train_pairs)
        test_rows = frame.pair_id.isin(test_pairs)
        target = (frame.channel == "SDM").astype(int)
        model = Pipeline([
            ("scale", StandardScaler()),
            ("classifier", LogisticRegression(max_iter=2500, class_weight="balanced", random_state=SEED)),
        ])
        model.fit(frame.loc[train_rows, columns], target.loc[train_rows])
        prediction = model.predict(frame.loc[test_rows, columns])
        rows.append({
            "protocol_version": PROTOCOL_VERSION,
            "protocol": protocol_label,
            "fold": fold,
            "held_out_groups": pair_groups.iloc[test_index][group_column].nunique(),
            "test_pairs": len(test_pairs),
            **metrics_dict(target.loc[test_rows], prediction),
        })
    return pd.DataFrame(rows)

split_pair_sets = {
    part: set(pair_meta.loc[pair_meta.split == part, "pair_id"])
    for part in ("train", "validation", "test")
}
assert split_pair_sets["train"].isdisjoint(split_pair_sets["validation"])
assert split_pair_sets["train"].isdisjoint(split_pair_sets["test"])
assert split_pair_sets["validation"].isdisjoint(split_pair_sets["test"])
print({
    part: {
        "speakers": pair_meta.loc[pair_meta.split == part, "speaker_id"].nunique(),
        "meetings": pair_meta.loc[pair_meta.split == part, "meeting_id"].nunique(),
        "pairs": len(split_pair_sets[part]),
    }
    for part in ("train", "validation", "test")
})


## 11. Recording-condition probe and predefined feature ablations

The primary logistic probe predicts IHM versus SDM for held-out speakers. Additional models are diagnostics, not a basis for selecting feature families after viewing test performance.

Very high recording-condition classification may be dominated by simple level or duration cues. Predefined feature ablations test whether condition identity remains recoverable from spectral and cepstral structure after obvious nuisance cues are removed. The raw, peak-normalized, and RMS-normalized waveform representations separately test whether gross amplitude differences explain recoverability.


In [ ]:
def metrics_dict(y_true, y_pred, score=None):
    result = {
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro"),
    }
    try:
        result["roc_auc"] = roc_auc_score(y_true, score) if score is not None and len(np.unique(y_true)) == 2 else np.nan
    except ValueError:
        result["roc_auc"] = np.nan
    return result

def make_classifier(name, scaler="standard"):
    scale = StandardScaler() if scaler == "standard" else RobustScaler() if scaler == "robust" else "passthrough"
    if name == "Logistic Regression":
        classifier = LogisticRegression(max_iter=2500, class_weight="balanced", random_state=SEED)
    elif name == "Linear SVM":
        classifier = SVC(kernel="linear", probability=True, class_weight="balanced", random_state=SEED)
    elif name == "Random Forest":
        classifier = RandomForestClassifier(n_estimators=300, class_weight="balanced", n_jobs=-1, random_state=SEED)
    else:
        raise ValueError(name)
    return Pipeline([("scale", scale), ("classifier", classifier)])

raw = feature_df[feature_df.representation == "raw"].copy()
raw["mic_target"] = (raw.channel == "SDM").astype(int)
train = raw.split == "train"
validation = raw.split == "validation"
test = raw.split == "test"
Xtr, ytr = raw.loc[train, feature_names], raw.loc[train, "mic_target"]
Xval, yval = raw.loc[validation, feature_names], raw.loc[validation, "mic_target"]
Xte, yte = raw.loc[test, feature_names], raw.loc[test, "mic_target"]

mic_rows, mic_models = [], {}
prediction_registry, prediction_clusters = {}, {}
for name in ("Logistic Regression", "Linear SVM", "Random Forest"):
    model = make_classifier(name)
    model.fit(Xtr, ytr)
    val_prediction = model.predict(Xval)
    prediction = model.predict(Xte)
    score = model.predict_proba(Xte)[:, 1]
    mic_rows.append({
        "protocol_version": PROTOCOL_VERSION,
        "experiment": "recording_condition_speaker_held_out",
        "model": name,
        "representation": "Raw",
        "validation_balanced_accuracy": balanced_accuracy_score(yval, val_prediction),
        **metrics_dict(yte, prediction, score),
    })
    mic_models[name] = model
    registry_name = f"recording_condition_{name}"
    prediction_registry[registry_name] = (yte.to_numpy(), prediction)
    prediction_clusters[registry_name] = {
        "speaker": raw.loc[test, "speaker_id"].to_numpy(),
        "meeting": raw.loc[test, "meeting_id"].to_numpy(),
    }

mic_results = pd.DataFrame(mic_rows).sort_values("validation_balanced_accuracy", ascending=False)
display(mic_results)
mic_results.to_csv(RESULTS_DIR / "baseline_results.csv", index=False)
best_mic_name = mic_results.iloc[0].model
best_mic_model = mic_models[best_mic_name]
ConfusionMatrixDisplay.from_predictions(
    yte, best_mic_model.predict(Xte), display_labels=["IHM", "SDM"], cmap="Blues", colorbar=False
)
plt.title(f"{best_mic_name}: speaker-held-out")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "microphone_confusion.png", dpi=200)
plt.show()

logistic = mic_models["Logistic Regression"]
coefficients = logistic.named_steps["classifier"].coef_[0]
important = np.argsort(np.abs(coefficients))[-20:]
fig, ax = plt.subplots(figsize=(8, 6))
colors = plt.cm.coolwarm((coefficients[important] - coefficients[important].min()) / (np.ptp(coefficients[important]) + 1e-9))
ax.barh(np.array(feature_names)[important], coefficients[important], color=colors)
ax.axvline(0, color="black", lw=0.8)
ax.set(xlabel="Standardized logistic coefficient (positive = SDM)", title="Features associated with recording condition")
plt.tight_layout()
fig.savefig(FIGURE_DIR / "microphone_coefficients.png", dpi=200)
plt.show()

ablation_rows = []
for representation in ("raw", "peak_norm", "rms_norm"):
    frame = feature_df[feature_df.representation == representation].copy()
    frame["mic_target"] = (frame.channel == "SDM").astype(int)
    for group_name, columns in FEATURE_GROUPS.items():
        tr, va, te = frame.split == "train", frame.split == "validation", frame.split == "test"
        model = make_classifier("Logistic Regression")
        model.fit(frame.loc[tr, columns], frame.loc[tr, "mic_target"])
        val_prediction = model.predict(frame.loc[va, columns])
        prediction = model.predict(frame.loc[te, columns])
        ablation_rows.append({
            "protocol_version": PROTOCOL_VERSION,
            "waveform_representation": representation,
            "feature_group": group_name,
            "n_features": len(columns),
            "validation_balanced_accuracy": balanced_accuracy_score(frame.loc[va, "mic_target"], val_prediction),
            "test_balanced_accuracy": balanced_accuracy_score(frame.loc[te, "mic_target"], prediction),
            "macro_f1": f1_score(frame.loc[te, "mic_target"], prediction, average="macro"),
        })
feature_ablation_results = pd.DataFrame(ablation_rows)
feature_ablation_results.to_csv(RESULTS_DIR / "feature_ablation_results.csv", index=False)
display(feature_ablation_results)

ablation_plot = feature_ablation_results.pivot(
    index="feature_group",
    columns="waveform_representation",
    values="test_balanced_accuracy",
)
ablation_plot = ablation_plot.sort_values("raw")
fig, ax = plt.subplots(figsize=(11, 6))
ablation_plot.plot.barh(ax=ax)
ax.axvline(0.5, color="black", ls="--", lw=0.8)
ax.set(
    xlim=(0, 1),
    xlabel="Speaker-held-out balanced accuracy",
    ylabel="Feature family",
    title="Recording-condition feature and amplitude ablations",
)
ax.legend(title="Waveform")
plt.tight_layout()
fig.savefig(FIGURE_DIR / "microphone_feature_ablation.png", dpi=200)
plt.show()

grouped_tables = []
for group_column, protocol_label in (("speaker_id", "Speaker-held-out"), ("meeting_id", "Meeting-held-out")):
    table = grouped_microphone_evaluation(raw, feature_names, group_column, protocol_label)
    if not table.empty:
        grouped_tables.append(table)
grouped_microphone_results = pd.concat(grouped_tables, ignore_index=True) if grouped_tables else pd.DataFrame()
grouped_microphone_results.to_csv(RESULTS_DIR / "grouped_microphone_results.csv", index=False)
display(grouped_microphone_results.groupby("protocol")[["balanced_accuracy", "macro_f1"]].agg(["mean", "std"]) if not grouped_microphone_results.empty else grouped_microphone_results)


## 12. Closed-set speaker-information probe

This is not speaker-independent speaker identification. The same known speaker identities occur in train, validation, and test, while utterance pair IDs are disjoint and both channels from each pair stay together.

- **Recording-condition probe:** evaluates held-out speakers.
- **Closed-set speaker-information probe:** evaluates held-out utterances from known speakers.

The task asks whether a representation retains information useful for distinguishing known speakers and whether that information transfers between close-talk and distant-microphone conditions.


In [ ]:
pair_counts = pair_meta.groupby("speaker_id").pair_id.nunique().sort_values(ascending=False)
eligible_speakers = []
for threshold in [20, 12, 8, 6]:
    eligible_speakers = pair_counts[pair_counts >= threshold].head(12).index.tolist()
    if len(eligible_speakers) >= 4:
        break

speaker_probe_available = len(eligible_speakers) >= 4
speaker_split_map = {}
speaker_pairs = set()
speaker_raw = pd.DataFrame()
if speaker_probe_available:
    generator = np.random.default_rng(SEED)
    for speaker in eligible_speakers:
        ids = pair_meta.loc[pair_meta.speaker_id == speaker, "pair_id"].to_numpy().copy()
        generator.shuffle(ids)
        n = len(ids)
        n_train, n_validation = max(2, int(0.70 * n)), max(1, int(0.15 * n))
        if n_train + n_validation >= n:
            n_train, n_validation = n - 2, 1
        speaker_split_map.update({pair_id: "train" for pair_id in ids[:n_train]})
        speaker_split_map.update({pair_id: "validation" for pair_id in ids[n_train:n_train + n_validation]})
        speaker_split_map.update({pair_id: "test" for pair_id in ids[n_train + n_validation:]})
    speaker_pairs = set(speaker_split_map)
    closed_set_parts = {
        part: {pair_id for pair_id, assigned in speaker_split_map.items() if assigned == part}
        for part in ("train", "validation", "test")
    }
    assert closed_set_parts["train"].isdisjoint(closed_set_parts["validation"])
    assert closed_set_parts["train"].isdisjoint(closed_set_parts["test"])
    assert closed_set_parts["validation"].isdisjoint(closed_set_parts["test"])
    assert all(
        pair_meta[pair_meta.pair_id.isin(closed_set_parts[part])].speaker_id.nunique() == len(eligible_speakers)
        for part in ("train", "validation", "test")
    )
    speaker_raw = raw[raw.pair_id.isin(speaker_pairs)].copy()
    speaker_raw["speaker_probe_split"] = speaker_raw.pair_id.map(speaker_split_map)
    assert speaker_raw.groupby("pair_id").speaker_probe_split.nunique().max() == 1
    print(f"Closed-set speaker-information probe: {len(eligible_speakers)} known speakers, {len(speaker_pairs)} disjoint pairs")
else:
    warnings.warn("Too few well-represented speakers; closed-set speaker and paired-alignment experiments will be skipped.")

def domain_probe(frame, label_col, split_column, source, target, model=None, columns=None):
    columns = feature_names if columns is None else list(columns)
    train_rows = (frame[split_column] == "train") & (frame.channel == source)
    test_rows = (frame[split_column] == "test") & (frame.channel == target)
    labels = set(frame.loc[train_rows, label_col]) & set(frame.loc[test_rows, label_col])
    train_rows &= frame[label_col].isin(labels)
    test_rows &= frame[label_col].isin(labels)
    train_pair_ids = set(frame.loc[train_rows, "pair_id"])
    test_pair_ids = set(frame.loc[test_rows, "pair_id"])
    assert train_pair_ids.isdisjoint(test_pair_ids)
    if len(labels) < 2 or train_rows.sum() < 10 or test_rows.sum() < 4:
        return None, None, None
    fitted = clone(model or make_classifier("Logistic Regression"))
    fitted.fit(frame.loc[train_rows, columns], frame.loc[train_rows, label_col])
    prediction = fitted.predict(frame.loc[test_rows, columns])
    return (
        metrics_dict(frame.loc[test_rows, label_col], prediction),
        fitted,
        (frame.loc[test_rows, label_col].to_numpy(), prediction),
    )

speaker_results, speaker_models = [], {}
if speaker_probe_available:
    for source, target in (("IHM", "IHM"), ("SDM", "SDM"), ("IHM", "SDM"), ("SDM", "IHM")):
        metrics, model, prediction = domain_probe(
            speaker_raw, "speaker_id", "speaker_probe_split", source, target
        )
        if metrics:
            condition = f"{source}→{target}"
            speaker_results.append({"protocol_version": PROTOCOL_VERSION, "condition": condition, **metrics})
            speaker_models[condition] = model
            prediction_registry[f"closed_set_speaker_{condition}"] = prediction

speaker_results = pd.DataFrame(speaker_results)
display(speaker_results)
if not speaker_results.empty:
    fig, ax = plt.subplots(figsize=(7, 4.5))
    ax.bar(speaker_results.condition, speaker_results.macro_f1)
    ax.set(ylim=(0, 1), ylabel="Macro-F1", title="Closed-set speaker information across conditions")
    plt.tight_layout()
    fig.savefig(FIGURE_DIR / "speaker_cross_channel.png", dpi=200)
    plt.show()
    same = speaker_results[speaker_results.condition.isin(["IHM→IHM", "SDM→SDM"])].macro_f1.mean()
    cross = speaker_results[speaker_results.condition.isin(["IHM→SDM", "SDM→IHM"])].macro_f1.mean()
    print(f"Mean same-condition speaker F1={same:.3f}; cross-condition F1={cross:.3f}; change={cross - same:+.3f}.")

def meeting_grouped_downstream_probe(frame, label_col, task_label, max_splits=5):
    pair_groups = frame[["pair_id", "meeting_id"]].drop_duplicates()
    n_meetings = pair_groups.meeting_id.nunique()
    if n_meetings < 3:
        return pd.DataFrame()
    splitter = GroupKFold(n_splits=min(max_splits, n_meetings))
    rows = []
    for fold, (train_index, test_index) in enumerate(
        splitter.split(pair_groups, groups=pair_groups.meeting_id), start=1
    ):
        train_pairs = set(pair_groups.iloc[train_index].pair_id)
        test_pairs = set(pair_groups.iloc[test_index].pair_id)
        assert train_pairs.isdisjoint(test_pairs)
        folded = frame.copy()
        folded["meeting_probe_split"] = np.where(
            folded.pair_id.isin(train_pairs),
            "train",
            np.where(folded.pair_id.isin(test_pairs), "test", "unused"),
        )
        if task_label == "small_lexical_probe":
            fold_training = folded[folded.meeting_probe_split == "train"].drop_duplicates("pair_id")
            fold_training = fold_training[
                fold_training[label_col].astype(str).str.match(r"^[a-z']+$", na=False)
            ]
            fold_stats = fold_training.groupby(label_col).agg(
                pairs=("pair_id", "nunique"), speakers=("speaker_id", "nunique")
            )
            fold_labels = (
                fold_stats[(fold_stats.pairs >= MIN_WORD_COUNT) & (fold_stats.speakers >= 3)]
                .sort_values("pairs", ascending=False)
                .head(TOP_K_WORDS)
                .index
            )
            if len(fold_labels) < 2:
                continue
            folded = folded[folded[label_col].isin(fold_labels)].copy()
        for source, target in (("IHM", "IHM"), ("SDM", "SDM"), ("IHM", "SDM"), ("SDM", "IHM")):
            metrics, _, _ = domain_probe(
                folded, label_col, "meeting_probe_split", source, target,
                make_classifier("Logistic Regression"), feature_names
            )
            if metrics:
                rows.append({
                    "protocol_version": PROTOCOL_VERSION,
                    "task": task_label,
                    "protocol": "Meeting-held-out",
                    "fold": fold,
                    "condition": f"{source}→{target}",
                    "held_out_meetings": pair_groups.iloc[test_index].meeting_id.nunique(),
                    **metrics,
                })
    return pd.DataFrame(rows)

meeting_downstream_tables = []
if speaker_probe_available:
    speaker_meeting_results = meeting_grouped_downstream_probe(
        raw, "speaker_id", "closed_set_speaker_information"
    )
    if not speaker_meeting_results.empty:
        meeting_downstream_tables.append(speaker_meeting_results)


## 13. Small lexical probe across recording conditions

This deliberately narrow experiment uses frequent single-word transcripts that occur across held-out speakers. It is a **small lexical probe**, not a broad lexical-classification benchmark.

All four directions are retained. The notebook reports IHM→SDM degradation relative to IHM→IHM and SDM→IHM degradation relative to SDM→SDM before any averaging. Directional asymmetry is described only when the absolute difference between the two drops reaches the declared LEXICAL_ASYMMETRY_THRESHOLD; no causal explanation is asserted.


In [ ]:
pair_meta["word_label"] = pair_meta.normalized_text.str.lower()
training_meta = pair_meta[pair_meta.split == "train"].copy()
single = training_meta.word_label.str.match(r"^[a-z']+$", na=False)
word_stats = training_meta[single].groupby("word_label").agg(
    pairs=("pair_id", "nunique"), speakers=("speaker_id", "nunique")
)
word_labels, used_word_threshold = [], None
for threshold in [MIN_WORD_COUNT, 10, 5]:
    candidates = (
        word_stats[(word_stats.pairs >= threshold) & (word_stats.speakers >= 3)]
        .sort_values("pairs", ascending=False)
        .head(TOP_K_WORDS)
        .index.tolist()
    )
    if len(candidates) >= 2:
        word_labels, used_word_threshold = candidates, threshold
        break

lexical_available = len(word_labels) >= 2
lexical_results, lexical_models = [], {}
lexical_raw = pd.DataFrame()
if lexical_available:
    lexical_raw = raw.merge(pair_meta[["pair_id", "word_label"]], on="pair_id", how="left", validate="many_to_one")
    lexical_raw = lexical_raw[lexical_raw.word_label.isin(word_labels)].copy()
    print(f"Small lexical probe: {len(word_labels)} words at minimum count {used_word_threshold}: {word_labels}")
    for source, target in (("IHM", "IHM"), ("SDM", "SDM"), ("IHM", "SDM"), ("SDM", "IHM")):
        metrics, model, prediction = domain_probe(lexical_raw, "word_label", "split", source, target)
        if metrics:
            condition = f"{source}→{target}"
            lexical_results.append({"protocol_version": PROTOCOL_VERSION, "condition": condition, **metrics})
            lexical_models[condition] = model
            prediction_registry[f"small_lexical_{condition}"] = prediction
    lexical_results = pd.DataFrame(lexical_results)
    display(lexical_results)
    if not lexical_results.empty:
        fig, ax = plt.subplots(figsize=(7, 4.5))
        ax.bar(lexical_results.condition, lexical_results.macro_f1)
        ax.set(ylim=(0, 1), ylabel="Macro-F1", title="Small lexical probe: speaker-held-out")
        plt.tight_layout()
        fig.savefig(FIGURE_DIR / "lexical_cross_channel.png", dpi=200)
        plt.show()
        directional = lexical_results.set_index("condition").macro_f1
        required_directions = {"IHM→IHM", "SDM→SDM", "IHM→SDM", "SDM→IHM"}
        if required_directions.issubset(directional.index):
            drop_ihm_to_sdm = directional["IHM→IHM"] - directional["IHM→SDM"]
            drop_sdm_to_ihm = directional["SDM→SDM"] - directional["SDM→IHM"]
            print(f"IHM→SDM degradation relative to IHM→IHM: {drop_ihm_to_sdm:+.3f}")
            print(f"SDM→IHM degradation relative to SDM→SDM: {drop_sdm_to_ihm:+.3f}")
            if abs(drop_ihm_to_sdm - drop_sdm_to_ihm) >= LEXICAL_ASYMMETRY_THRESHOLD:
                print(
                    "Cross-condition transfer is asymmetric in this run. This may reflect differences between "
                    "learning from close-talk and distant speech, but a dedicated experiment is needed to isolate the cause."
                )
else:
    lexical_results = pd.DataFrame()
    print("Small lexical probe skipped: fewer than two viable single-word classes across held-out speaker splits.")

if lexical_available:
    lexical_meeting_frame = raw.merge(
        pair_meta[["pair_id", "word_label"]], on="pair_id", how="left", validate="many_to_one"
    )
    lexical_meeting_results = meeting_grouped_downstream_probe(
        lexical_meeting_frame, "word_label", "small_lexical_probe"
    )
    if not lexical_meeting_results.empty:
        meeting_downstream_tables.append(lexical_meeting_results)

meeting_grouped_downstream_results = (
    pd.concat(meeting_downstream_tables, ignore_index=True)
    if meeting_downstream_tables else pd.DataFrame()
)
meeting_grouped_downstream_results.to_csv(
    RESULTS_DIR / "meeting_grouped_downstream_results.csv", index=False
)
if meeting_grouped_downstream_results.empty:
    print("Meeting-held-out downstream probes skipped: too few meetings or too few overlapping labels per fold.")
else:
    display(meeting_grouped_downstream_results)


## 14. Source-only preprocessing versus target-aware adaptation

**Source-only methods:** Raw, MFCC-CMN, MFCC-CMVN, Robust scaling, PCA whitening, and the pair-aligned representation introduced later.

**Target-aware method:** CORAL (UDA), which estimates target-channel covariance from unlabeled target recordings in the training partition.

MFCC-CMN and MFCC-CMVN normalize only the cepstral part of the combined representation; non-cepstral features remain unchanged. The accurate claim is therefore that MFCC-CMVN alone may not remove recording-condition information from the full combined representation.

CORAL asks a different question: if unlabeled data from the target recording condition is available, can covariance alignment recover cross-condition performance? It is reported separately rather than presented as assumption-equivalent to source-only methods.


In [ ]:
def pipeline_for(method):
    if method in ("Raw", "MFCC-CMN", "MFCC-CMVN"):
        preprocessing = StandardScaler()
    elif method == "Robust":
        preprocessing = RobustScaler()
    elif method == "PCA whiten":
        preprocessing = Pipeline([
            ("scale", StandardScaler()),
            ("pca", PCA(n_components=0.95, whiten=True, random_state=SEED)),
        ])
    else:
        raise ValueError(method)
    return Pipeline([
        ("preprocess", preprocessing),
        ("classifier", LogisticRegression(max_iter=2500, class_weight="balanced", random_state=SEED)),
    ])

def eval_mic_method(method, representation):
    frame = feature_df[feature_df.representation == representation].copy()
    frame["target"] = (frame.channel == "SDM").astype(int)
    train_rows, validation_rows, test_rows = frame.split == "train", frame.split == "validation", frame.split == "test"
    model = pipeline_for(method)
    model.fit(frame.loc[train_rows, feature_names], frame.loc[train_rows, "target"])
    validation_prediction = model.predict(frame.loc[validation_rows, feature_names])
    prediction = model.predict(frame.loc[test_rows, feature_names])
    score = model.predict_proba(frame.loc[test_rows, feature_names])[:, 1]
    return (
        metrics_dict(frame.loc[test_rows, "target"], prediction, score),
        balanced_accuracy_score(frame.loc[validation_rows, "target"], validation_prediction),
        model,
    )

def eval_task_method(method, representation, task):
    base = feature_df[feature_df.representation == representation].copy()
    label_col, split_col = ("word_label", "split") if task == "lexical" else ("speaker_id", "speaker_probe_split")
    if task == "speaker":
        if not speaker_probe_available:
            return np.nan, np.nan
        base = base[base.pair_id.isin(speaker_pairs)].copy()
        base["speaker_probe_split"] = base.pair_id.map(speaker_split_map)
    else:
        if not lexical_available:
            return np.nan, np.nan
        base = base.merge(pair_meta[["pair_id", "word_label"]], on="pair_id", how="left", validate="many_to_one")
        base = base[base.word_label.isin(word_labels)]
    scores = {}
    for source, target in (("IHM", "IHM"), ("SDM", "SDM"), ("IHM", "SDM"), ("SDM", "IHM")):
        metrics, _, _ = domain_probe(
            base, label_col, split_col, source, target, pipeline_for(method), feature_names
        )
        scores[f"{source}→{target}"] = metrics["macro_f1"] if metrics else np.nan
    same = np.nanmean([scores["IHM→IHM"], scores["SDM→SDM"]])
    cross = np.nanmean([scores["IHM→SDM"], scores["SDM→IHM"]])
    return same, cross

method_specs = (
    ("Raw", "raw"),
    ("MFCC-CMN", "mfcc_cmn"),
    ("MFCC-CMVN", "mfcc_cmvn"),
    ("Robust", "raw"),
    ("PCA whiten", "raw"),
)
normalization_rows, normalization_models = [], {}
for method, representation in method_specs:
    mic_metrics, validation_mic, model = eval_mic_method(method, representation)
    speaker_same, speaker_cross = eval_task_method(method, representation, "speaker")
    lexical_same, lexical_cross = eval_task_method(method, representation, "lexical")
    normalization_rows.append({
        "protocol_version": PROTOCOL_VERSION,
        "assumption": "source_only",
        "representation": method,
        "validation_mic_accuracy": validation_mic,
        "mic_accuracy": mic_metrics["balanced_accuracy"],
        "mic_f1": mic_metrics["macro_f1"],
        "speaker_same_channel_f1": speaker_same,
        "speaker_cross_channel_f1": speaker_cross,
        "lexical_same_channel_f1": lexical_same,
        "lexical_cross_channel_f1": lexical_cross,
    })
    normalization_models[method] = model

def matrix_power(covariance, power, eps=1e-5):
    values, vectors = np.linalg.eigh(covariance)
    return (vectors * np.maximum(values, eps) ** power) @ vectors.T

def coral_fit(source, target):
    source_mean, target_mean = source.mean(0), target.mean(0)
    source_cov = np.cov(source, rowvar=False) + 1e-3 * np.eye(source.shape[1])
    target_cov = np.cov(target, rowvar=False) + 1e-3 * np.eye(source.shape[1])
    return source_mean, target_mean, matrix_power(source_cov, -0.5) @ matrix_power(target_cov, 0.5)

coral_cross = []
if speaker_probe_available:
    base = raw[raw.pair_id.isin(speaker_pairs)].copy()
    base["speaker_probe_split"] = base.pair_id.map(speaker_split_map)
    for source, target in (("IHM", "SDM"), ("SDM", "IHM")):
        source_train = (base.speaker_probe_split == "train") & (base.channel == source)
        unlabeled_target_train = (base.speaker_probe_split == "train") & (base.channel == target)
        target_test = (base.speaker_probe_split == "test") & (base.channel == target)
        assert set(base.loc[unlabeled_target_train, "pair_id"]).issubset(
            {pair_id for pair_id, split in speaker_split_map.items() if split == "train"}
        )
        assert set(base.loc[source_train, "pair_id"]).isdisjoint(set(base.loc[target_test, "pair_id"]))
        scaler = StandardScaler().fit(base.loc[source_train, feature_names])
        source_x = scaler.transform(base.loc[source_train, feature_names])
        target_adapt_x = scaler.transform(base.loc[unlabeled_target_train, feature_names])
        target_test_x = scaler.transform(base.loc[target_test, feature_names])
        source_mean, target_mean, transform = coral_fit(source_x, target_adapt_x)
        mapped_source = (source_x - source_mean) @ transform + target_mean
        classifier = LogisticRegression(max_iter=2500, class_weight="balanced", random_state=SEED)
        classifier.fit(mapped_source, base.loc[source_train, "speaker_id"])
        coral_cross.append(
            f1_score(base.loc[target_test, "speaker_id"], classifier.predict(target_test_x), average="macro")
        )

normalization_rows.append({
    "protocol_version": PROTOCOL_VERSION,
    "assumption": "target_aware_unlabeled",
    "representation": "CORAL (UDA)",
    "validation_mic_accuracy": np.nan,
    "mic_accuracy": np.nan,
    "mic_f1": np.nan,
    "speaker_same_channel_f1": np.nan,
    "speaker_cross_channel_f1": np.mean(coral_cross) if coral_cross else np.nan,
    "lexical_same_channel_f1": np.nan,
    "lexical_cross_channel_f1": np.nan,
})
normalization_results = pd.DataFrame(normalization_rows)
normalization_results.to_csv(RESULTS_DIR / "normalization_results.csv", index=False)
display(normalization_results)

source_only_results = normalization_results[normalization_results.assumption == "source_only"]
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].bar(source_only_results.representation, source_only_results.mic_accuracy)
axes[0].axhline(0.5, color="black", ls="--", lw=0.8)
axes[0].set(ylim=(0, 1), ylabel="Balanced accuracy", title="Source-only recording-condition recoverability")
axes[0].tick_params(axis="x", rotation=25)
axes[1].bar(source_only_results.representation, source_only_results.speaker_cross_channel_f1)
axes[1].set(ylim=(0, 1), ylabel="Macro-F1", title="Source-only cross-condition speaker probe")
axes[1].tick_params(axis="x", rotation=25)
plt.tight_layout()
fig.savefig(FIGURE_DIR / "normalization_tradeoff.png", dpi=200)
plt.show()
coral_row = normalization_results[normalization_results.representation == "CORAL (UDA)"]
if not coral_row.empty:
    display(Markdown("**Target-aware CORAL (UDA)** is reported separately because it uses unlabeled target-condition training data."))
    display(coral_row)

best_norm = source_only_results.sort_values("validation_mic_accuracy", ascending=True).iloc[0]
raw_norm = source_only_results[source_only_results.representation == "Raw"].iloc[0]
method_to_rep = {"Raw": "raw", "MFCC-CMN": "mfcc_cmn", "MFCC-CMVN": "mfcc_cmvn", "Robust": "raw", "PCA whiten": "raw"}
for label, method in (("raw", "Raw"), ("best_source_only", best_norm.representation)):
    frame = feature_df[feature_df.representation == method_to_rep[method]].copy()
    frame["target"] = (frame.channel == "SDM").astype(int)
    test_rows = frame.split == "test"
    registry_name = f"{label}_recording_condition"
    prediction_registry[registry_name] = (
        frame.loc[test_rows, "target"].to_numpy(),
        normalization_models[method].predict(frame.loc[test_rows, feature_names]),
    )
    prediction_clusters[registry_name] = {
        "speaker": frame.loc[test_rows, "speaker_id"].to_numpy(),
        "meeting": frame.loc[test_rows, "meeting_id"].to_numpy(),
    }


## 15. Paired representation alignment

The pair-aligned encoder sees engineered features and is trained only on closed-set training pairs. Its auxiliary neural head predicts known speaker identity while a cosine penalty encourages matched IHM/SDM embeddings to move closer.

This is an alignment objective, not proof of invariance. For every lambda, the encoder is frozen and evaluated with newly fitted logistic probes:

- a recording-condition probe trained on training embeddings;
- four closed-set speaker-information directions: IHM→IHM, SDM→SDM, IHM→SDM, and SDM→IHM;
- matched-pair cosine distance as a separate geometric quantity.

Lambda selection uses validation embeddings only. The test split is never used to select lambda. Neural task-head performance remains a training diagnostic and is not used in the central representation comparison.


In [ ]:
from torch import nn
import torch.nn.functional as F

class PairAlignedEncoder(nn.Module):
    def __init__(self, input_dim, n_classes):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.15),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
        )
        self.task_head = nn.Linear(32, n_classes)

    def forward(self, x):
        embedding = self.encoder(x)
        return self.task_head(embedding), embedding

def pair_alignment_loss(a, b):
    return (1 - F.cosine_similarity(a, b)).mean()

def embedding_speaker_probe(train_h, train_d, train_y, eval_h, eval_d, eval_y):
    embeddings = {"IHM": (train_h, eval_h), "SDM": (train_d, eval_d)}
    scores, predictions = {}, {}
    for source, target in (("IHM", "IHM"), ("SDM", "SDM"), ("IHM", "SDM"), ("SDM", "IHM")):
        classifier = Pipeline([
            ("scale", StandardScaler()),
            ("classifier", LogisticRegression(max_iter=2500, class_weight="balanced", random_state=SEED)),
        ])
        classifier.fit(embeddings[source][0], train_y)
        prediction = classifier.predict(embeddings[target][1])
        direction = f"{source}→{target}"
        scores[direction] = f1_score(eval_y, prediction, average="macro")
        predictions[direction] = prediction
    scores["mean_same"] = np.mean([scores["IHM→IHM"], scores["SDM→SDM"]])
    scores["mean_cross"] = np.mean([scores["IHM→SDM"], scores["SDM→IHM"]])
    return scores, predictions

alignment_results = pd.DataFrame()
embedding_payloads = {}
if speaker_probe_available:
    aligned_frame = feature_df[
        (feature_df.representation == "raw") & feature_df.pair_id.isin(speaker_pairs)
    ].copy()
    aligned_frame["probe_split"] = aligned_frame.pair_id.map(speaker_split_map)
    label_map = {speaker: index for index, speaker in enumerate(sorted(eligible_speakers))}
    vectors = {
        (row.pair_id, row.channel): row[feature_names].to_numpy(dtype=np.float32)
        for _, row in aligned_frame.iterrows()
    }
    pair_label = {
        row.pair_id: label_map[row.speaker_id]
        for row in pair_meta[pair_meta.pair_id.isin(speaker_pairs)].itertuples()
    }
    ids_by_split = {
        part: [
            pair_id for pair_id, assigned in speaker_split_map.items()
            if assigned == part and (pair_id, "IHM") in vectors and (pair_id, "SDM") in vectors
        ]
        for part in ("train", "validation", "test")
    }
    assert set(ids_by_split["train"]).isdisjoint(ids_by_split["validation"])
    assert set(ids_by_split["train"]).isdisjoint(ids_by_split["test"])
    assert set(ids_by_split["validation"]).isdisjoint(ids_by_split["test"])

    fit_vectors = np.vstack([
        vectors[(pair_id, channel)]
        for pair_id in ids_by_split["train"]
        for channel in ("IHM", "SDM")
    ])
    alignment_scaler = StandardScaler().fit(fit_vectors)

    def batch_arrays(ids):
        return (
            alignment_scaler.transform(np.vstack([vectors[(pair_id, "IHM")] for pair_id in ids])).astype("float32"),
            alignment_scaler.transform(np.vstack([vectors[(pair_id, "SDM")] for pair_id in ids])).astype("float32"),
            np.array([pair_label[pair_id] for pair_id in ids], dtype="int64"),
        )

    arrays = {part: batch_arrays(ids) for part, ids in ids_by_split.items()}

    def embed_split(model, part):
        ihm_x, sdm_x, labels = arrays[part]
        model.eval()
        with torch.no_grad():
            ihm_logits, ihm_embedding = model(torch.tensor(ihm_x, device=device))
            sdm_logits, sdm_embedding = model(torch.tensor(sdm_x, device=device))
        return (
            ihm_embedding.cpu().numpy(),
            sdm_embedding.cpu().numpy(),
            labels,
            ihm_logits.argmax(1).cpu().numpy(),
            sdm_logits.argmax(1).cpu().numpy(),
        )

    rows = []
    for lambda_pair in tqdm(LAMBDA_VALUES, desc="Paired-alignment sweep"):
        torch.manual_seed(SEED)
        model = PairAlignedEncoder(len(feature_names), len(label_map)).to(device)
        optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
        best_state, best_validation_loss, stale = None, np.inf, 0
        generator = np.random.default_rng(SEED)
        for epoch in range(40 if RUN_MODE != "smoke" else 20):
            model.train()
            order = generator.permutation(len(ids_by_split["train"]))
            ihm_x, sdm_x, labels = arrays["train"]
            for start in range(0, len(order), 64):
                indices = order[start:start + 64]
                ihm_batch = torch.tensor(ihm_x[indices], device=device)
                sdm_batch = torch.tensor(sdm_x[indices], device=device)
                target = torch.tensor(labels[indices], device=device)
                ihm_logits, ihm_embedding = model(ihm_batch)
                sdm_logits, sdm_embedding = model(sdm_batch)
                loss = (
                    F.cross_entropy(ihm_logits, target) + F.cross_entropy(sdm_logits, target)
                ) / 2 + lambda_pair * pair_alignment_loss(ihm_embedding, sdm_embedding)
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            val_h, val_d, val_y = arrays["validation"]
            model.eval()
            with torch.no_grad():
                ihm_logits, ihm_embedding = model(torch.tensor(val_h, device=device))
                sdm_logits, sdm_embedding = model(torch.tensor(val_d, device=device))
                validation_loss = float((
                    (F.cross_entropy(ihm_logits, torch.tensor(val_y, device=device))
                     + F.cross_entropy(sdm_logits, torch.tensor(val_y, device=device))) / 2
                    + lambda_pair * pair_alignment_loss(ihm_embedding, sdm_embedding)
                ).cpu())
            if validation_loss < best_validation_loss - 1e-4:
                best_validation_loss = validation_loss
                best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
                stale = 0
            else:
                stale += 1
                if stale >= 6:
                    break

        model.load_state_dict(best_state)
        train_h, train_d, train_y, train_head_h, train_head_d = embed_split(model, "train")
        val_h, val_d, val_y, val_head_h, val_head_d = embed_split(model, "validation")
        test_h, test_d, test_y, test_head_h, test_head_d = embed_split(model, "test")

        train_embeddings = np.vstack([train_h, train_d])
        train_condition = np.r_[np.zeros(len(train_h)), np.ones(len(train_d))]
        condition_probe = Pipeline([
            ("scale", StandardScaler()),
            ("classifier", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=SEED)),
        ])
        condition_probe.fit(train_embeddings, train_condition)

        validation_embeddings = np.vstack([val_h, val_d])
        validation_condition = np.r_[np.zeros(len(val_h)), np.ones(len(val_d))]
        test_embeddings = np.vstack([test_h, test_d])
        test_condition = np.r_[np.zeros(len(test_h)), np.ones(len(test_d))]
        validation_condition_prediction = condition_probe.predict(validation_embeddings)
        test_condition_prediction = condition_probe.predict(test_embeddings)

        validation_speaker, _ = embedding_speaker_probe(
            train_h, train_d, train_y, val_h, val_d, val_y
        )
        test_speaker, test_speaker_predictions = embedding_speaker_probe(
            train_h, train_d, train_y, test_h, test_d, test_y
        )
        validation_neural_task_f1 = f1_score(
            np.r_[val_y, val_y], np.r_[val_head_h, val_head_d], average="macro"
        )
        neural_task_f1 = f1_score(
            np.r_[test_y, test_y], np.r_[test_head_h, test_head_d], average="macro"
        )
        validation_mic = balanced_accuracy_score(
            validation_condition, validation_condition_prediction
        )
        test_mic = balanced_accuracy_score(test_condition, test_condition_prediction)
        matched_distance = np.mean(rowwise_cosine_distance(test_h, test_d))
        permutation = np.roll(np.arange(len(test_d)), 1) if len(test_d) > 1 else np.arange(len(test_d))
        unmatched_distance = np.mean(rowwise_cosine_distance(test_h, test_d[permutation]))
        selection_score = validation_speaker["mean_cross"] - 0.20 * abs(validation_mic - 0.5)

        rows.append({
            "protocol_version": PROTOCOL_VERSION,
            "lambda_pair": lambda_pair,
            "validation_frozen_speaker_cross_channel_f1": validation_speaker["mean_cross"],
            "validation_mic_probe_accuracy": validation_mic,
            "selection_score": selection_score,
            "speaker_same_channel_f1": test_speaker["mean_same"],
            "speaker_cross_channel_f1": test_speaker["mean_cross"],
            "neural_task_head_f1": neural_task_f1,
            "validation_neural_task_head_f1": validation_neural_task_f1,
            "mic_probe_accuracy": test_mic,
            "matched_pair_distance": matched_distance,
            "unmatched_distance": unmatched_distance,
            "epochs": epoch + 1,
        })
        embedding_payloads[lambda_pair] = {
            "model": model,
            "train_h": train_h,
            "train_d": train_d,
            "test_h": test_h,
            "test_d": test_d,
            "Z_train": train_embeddings,
            "Z_test": test_embeddings,
            "mic_truth": test_condition,
            "mic_prediction": test_condition_prediction,
            "speaker_truth": np.r_[test_y, test_y],
            "speaker_predictions": test_speaker_predictions,
            "pair_ids": np.r_[ids_by_split["test"], ids_by_split["test"]],
        }

    alignment_results = pd.DataFrame(rows)
    lambda_zero = alignment_results.iloc[(alignment_results.lambda_pair - 0.0).abs().argmin()]
    denominator = lambda_zero.matched_pair_distance
    alignment_results["pair_distance_reduction_percent"] = np.where(
        abs(denominator) > 1e-12,
        100 * (lambda_zero.matched_pair_distance - alignment_results.matched_pair_distance) / denominator,
        np.nan,
    )
    alignment_results["mic_probe_reduction_points"] = (
        lambda_zero.mic_probe_accuracy - alignment_results.mic_probe_accuracy
    )
    alignment_results["task_f1_change"] = (
        alignment_results.speaker_cross_channel_f1 - lambda_zero.speaker_cross_channel_f1
    )
    alignment_results.to_csv(RESULTS_DIR / "pair_alignment_results.csv", index=False)
    display(alignment_results[[
        "lambda_pair", "speaker_cross_channel_f1", "mic_probe_accuracy",
        "matched_pair_distance", "pair_distance_reduction_percent",
        "mic_probe_reduction_points", "task_f1_change"
    ]])

    best_lambda = float(
        alignment_results.sort_values("selection_score", ascending=False).iloc[0].lambda_pair
    )
    best_embedding = embedding_payloads[best_lambda]
    prediction_registry["pair_aligned_recording_condition"] = (
        best_embedding["mic_truth"], best_embedding["mic_prediction"]
    )
    test_meta = pair_meta.set_index("pair_id").loc[ids_by_split["test"]]
    prediction_clusters["pair_aligned_recording_condition"] = {
        "speaker": np.r_[test_meta.speaker_id.to_numpy(), test_meta.speaker_id.to_numpy()],
        "meeting": np.r_[test_meta.meeting_id.to_numpy(), test_meta.meeting_id.to_numpy()],
    }
    print(f"Validation-selected pair-alignment weight: λ={best_lambda:g}.")
else:
    print("Paired-alignment model skipped because the closed-set speaker-information task is unavailable.")


## 16. Standardized frozen-probe comparison

All points in the central tradeoff plot now use the same frozen logistic-probe protocol, making the comparison across representations directly interpretable.

Every representation uses the same closed-set pair subset, disjoint train/test pair IDs, source-channel speaker-probe directions, and train-fitted preprocessing. Pair-aligned encoders are trained only on training pairs and frozen before probing. The x-axis is recording-condition probe balanced accuracy; the y-axis is mean cross-condition closed-set speaker-probe macro-F1.

The jointly trained neural task head is excluded from this comparison. CORAL is also excluded because its target-aware assumption differs from the source-only and pair-aligned points.


In [ ]:
def evaluate_frozen_classical_representation(method, representation):
    frame = feature_df[
        (feature_df.representation == representation) & feature_df.pair_id.isin(speaker_pairs)
    ].copy()
    frame["probe_split"] = frame.pair_id.map(speaker_split_map)
    frame["condition_target"] = (frame.channel == "SDM").astype(int)
    train_rows, test_rows = frame.probe_split == "train", frame.probe_split == "test"
    condition_model = pipeline_for(method)
    condition_model.fit(frame.loc[train_rows, feature_names], frame.loc[train_rows, "condition_target"])
    condition_prediction = condition_model.predict(frame.loc[test_rows, feature_names])
    direction_scores = {}
    for source, target in (("IHM", "IHM"), ("SDM", "SDM"), ("IHM", "SDM"), ("SDM", "IHM")):
        metrics, _, _ = domain_probe(
            frame, "speaker_id", "probe_split", source, target, pipeline_for(method), feature_names
        )
        direction_scores[f"{source}→{target}"] = metrics["macro_f1"] if metrics else np.nan
    return {
        "protocol_version": PROTOCOL_VERSION,
        "representation": method,
        "representation_family": "classical_source_only",
        "mic_probe_accuracy": balanced_accuracy_score(
            frame.loc[test_rows, "condition_target"], condition_prediction
        ),
        "speaker_same_channel_f1": np.nanmean([
            direction_scores["IHM→IHM"], direction_scores["SDM→SDM"]
        ]),
        "speaker_cross_channel_f1": np.nanmean([
            direction_scores["IHM→SDM"], direction_scores["SDM→IHM"]
        ]),
    }

standardized_rows = []
if speaker_probe_available:
    for method, representation in method_specs:
        standardized_rows.append(evaluate_frozen_classical_representation(method, representation))
    for row in alignment_results.itertuples():
        standardized_rows.append({
            "protocol_version": PROTOCOL_VERSION,
            "representation": f"PairAlign λ={row.lambda_pair:g}",
            "representation_family": "pair_aligned",
            "mic_probe_accuracy": row.mic_probe_accuracy,
            "speaker_same_channel_f1": row.speaker_same_channel_f1,
            "speaker_cross_channel_f1": row.speaker_cross_channel_f1,
        })

standardized_probe_results = pd.DataFrame(standardized_rows)
standardized_probe_results.to_csv(RESULTS_DIR / "standardized_frozen_probe_results.csv", index=False)
display(standardized_probe_results)

if not standardized_probe_results.empty:
    fig, ax = plt.subplots(figsize=(9, 6))
    for row in standardized_probe_results.itertuples():
        marker = "o" if row.representation_family == "classical_source_only" else "s"
        ax.scatter(row.mic_probe_accuracy, row.speaker_cross_channel_f1, s=60, marker=marker)
        ax.annotate(
            row.representation,
            (row.mic_probe_accuracy, row.speaker_cross_channel_f1),
            xytext=(5, 5),
            textcoords="offset points",
            fontsize=8,
        )
    ax.axvline(0.5, color="black", ls="--", lw=0.8)
    ax.set(
        xlabel="Frozen recording-condition probe balanced accuracy ↓",
        ylabel="Frozen cross-condition speaker-probe macro-F1 ↑",
        title="MicCheck standardized frozen-probe tradeoff",
        xlim=(0.45, 1.02),
        ylim=(0, 1.02),
    )
    plt.tight_layout()
    fig.savefig(FIGURE_DIR / "central_tradeoff.png", dpi=240)
    plt.show()

if speaker_probe_available and not alignment_results.empty:
    test_sp = speaker_raw[speaker_raw.speaker_probe_split == "test"].copy()
    train_sp = speaker_raw[speaker_raw.speaker_probe_split == "train"]
    raw_scale = StandardScaler().fit(train_sp[feature_names])
    raw_pca = PCA(2, random_state=SEED).fit(raw_scale.transform(train_sp[feature_names]))
    raw_xy = raw_pca.transform(raw_scale.transform(test_sp[feature_names]))
    aligned_pca = PCA(2, random_state=SEED).fit(best_embedding["Z_train"])
    aligned_xy = aligned_pca.transform(best_embedding["Z_test"])
    fig, axes = plt.subplots(2, 2, figsize=(13, 10))
    panels = (
        (axes[0, 0], raw_xy, test_sp.channel.to_numpy(), "Raw by condition"),
        (axes[0, 1], aligned_xy, np.where(best_embedding["mic_truth"] == 0, "IHM", "SDM"), "Pair-aligned by condition"),
        (axes[1, 0], raw_xy, test_sp.speaker_id.to_numpy(), "Raw by speaker"),
        (axes[1, 1], aligned_xy, best_embedding["speaker_truth"], "Pair-aligned by speaker"),
    )
    for ax, coordinates, labels, title in panels:
        for label in np.unique(labels):
            mask = labels == label
            ax.scatter(coordinates[mask, 0], coordinates[mask, 1], s=14, alpha=0.65, label=str(label))
        ax.set(title=title, xlabel="PC1", ylabel="PC2")
        if len(np.unique(labels)) <= 12:
            ax.legend(fontsize=7, ncol=2)
    plt.tight_layout()
    fig.savefig(FIGURE_DIR / "pca_before_after.png", dpi=200)
    plt.show()

    fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
    axes[0].plot(alignment_results.lambda_pair, alignment_results.speaker_cross_channel_f1, "o-", label="frozen cross-condition speaker F1")
    axes[0].plot(alignment_results.lambda_pair, alignment_results.mic_probe_accuracy, "o-", label="frozen condition probe")
    axes[0].plot(alignment_results.lambda_pair, alignment_results.neural_task_head_f1, "o--", label="neural head diagnostic")
    axes[0].set(xlabel="Pair-alignment weight λ", ylabel="Score", title="Paired-alignment sweep", ylim=(0, 1))
    axes[0].legend(fontsize=8)
    axes[1].plot(alignment_results.lambda_pair, alignment_results.matched_pair_distance, "o-", label="matched")
    axes[1].plot(alignment_results.lambda_pair, alignment_results.unmatched_distance, "o-", label="unmatched")
    axes[1].set(xlabel="Pair-alignment weight λ", ylabel="Cosine distance", title="Embedding geometry")
    axes[1].legend()
    plt.tight_layout()
    fig.savefig(FIGURE_DIR / "pair_alignment_sweep.png", dpi=200)
    plt.show()

    fig, ax = plt.subplots(figsize=(7, 5))
    ax.scatter(alignment_results.matched_pair_distance, alignment_results.mic_probe_accuracy, s=65)
    for row in alignment_results.itertuples():
        ax.annotate(f"λ={row.lambda_pair:g}", (row.matched_pair_distance, row.mic_probe_accuracy), xytext=(5, 5), textcoords="offset points")
    ax.axhline(0.5, color="black", ls="--", lw=0.8)
    ax.set(
        xlabel="Mean matched IHM/SDM embedding cosine distance",
        ylabel="Frozen recording-condition probe balanced accuracy",
        title="Geometric pair alignment versus channel decodability",
        ylim=(0.45, 1.02),
    )
    plt.tight_layout()
    fig.savefig(FIGURE_DIR / "pair_distance_vs_channel_leakage.png", dpi=200)
    plt.show()


## 17. Exploratory counterfactual recording-condition transformation

An aggregate log-amplitude response is estimated from **training pairs only** using Welch spectra, median aggregation, and frequency smoothing. Applying the corresponding regularized FIR filter to held-out IHM speech creates a pseudo-SDM diagnostic.

This transformation changes spectral coloration but cannot recreate room geometry, reverberation, source position, noise, or hardware. Prediction shifts indicate sensitivity to the estimated coloration; they do not isolate a causal microphone effect.


In [ ]:
cf_train=pair_meta[pair_meta.split=="train"].sample(min(COUNTERFACTUAL_TRAIN_PAIRS,(pair_meta.split=="train").sum()),random_state=SEED)
cf_test=pair_meta[pair_meta.split=="test"].sample(min(3,(pair_meta.split=="test").sum()),random_state=SEED)
cf_psd={"IHM":{},"SDM":{}}; cf_audio={}
for config,column,channel in (("ihm","ihm_audio_id","IHM"),("sdm","sdm_audio_id","SDM")):
    combined=pd.concat([cf_train,cf_test]).drop_duplicates("pair_id"); wanted=dict(zip(combined[column],combined.pair_id)); test_ids=set(cf_test.pair_id)
    for audio_id,y,sr in stream_selected(config,wanted):
        pair_id=wanted[audio_id]; f,psd=signal.welch(y,sr,nperseg=1024); cf_psd[channel][pair_id]=psd
        if pair_id in test_ids: cf_audio[(pair_id,channel)]=y.copy()
        del y
training_differences=[]
for pair_id in cf_train.pair_id:
    if pair_id in cf_psd["IHM"] and pair_id in cf_psd["SDM"]:
        training_differences.append(.5*(np.log(cf_psd["SDM"][pair_id]+1e-10)-np.log(cf_psd["IHM"][pair_id]+1e-10)))
if not training_differences: raise RuntimeError("No training audio pairs available for counterfactual response")
log_gain=ndimage.gaussian_filter1d(np.median(np.vstack(training_differences),axis=0),sigma=3)
gain=np.exp(np.clip(log_gain,-3,3)); taps=signal.firwin2(513,f/(16000/2),gain)

def pseudo_sdm(y):
    out=signal.fftconvolve(y,taps,mode="same").astype("float32"); peak=np.max(np.abs(out),initial=0)
    return out*(.99/peak) if peak>.99 else out
def vector_for(y): return acoustic_variants(y,16000)["raw"]

cf_rows=[]; raw_mic_model=normalization_models["Raw"]
for row in cf_test.itertuples():
    if (row.pair_id,"IHM") not in cf_audio or (row.pair_id,"SDM") not in cf_audio: continue
    hi=cf_audio[(row.pair_id,"IHM")]; sd=cf_audio[(row.pair_id,"SDM")]; pseudo=pseudo_sdm(hi)
    vectors=np.vstack([vector_for(hi),vector_for(pseudo),vector_for(sd)])
    probabilities=raw_mic_model.predict_proba(pd.DataFrame(vectors,columns=feature_names))[:,1]
    record={"pair_id":row.pair_id,"speaker_id":row.speaker_id,"text":row.text,"p_sdm_ihm":probabilities[0],"p_sdm_pseudo":probabilities[1],"p_sdm_real":probabilities[2],"lexical_prediction_changed":np.nan}
    if lexical_available and row.word_label in word_labels and "IHM→SDM" in lexical_models:
        task_model=lexical_models["IHM→SDM"]; predictions=task_model.predict(pd.DataFrame(vectors,columns=feature_names)); record["lexical_prediction_changed"]=bool(predictions[1]!=predictions[0]); record["lexical_predictions"]=" / ".join(map(str,predictions))
    cf_rows.append(record)
    print(f"{row.meeting_id} | {row.speaker_id} | {row.text}\nOriginal IHM / pseudo-SDM / actual SDM")
    display(Audio(hi,rate=16000)); display(Audio(pseudo,rate=16000)); display(Audio(sd,rate=16000))
    fig,axes=plt.subplots(1,3,figsize=(15,3.8))
    for ax,y,label in zip(axes,(hi,pseudo,sd),("Original IHM","Pseudo-SDM","Actual SDM")):
        L=logmel(y); ax.imshow(L,origin="lower",aspect="auto",extent=[0,len(y)/16000,0,8000],cmap="magma"); ax.set(title=label,xlabel="Time (s)",ylabel="Frequency (Hz)")
    plt.tight_layout(); fig.savefig(FIGURE_DIR/f"counterfactual_{row.pair_id}.png",dpi=200); plt.show()
counterfactual_results=pd.DataFrame(cf_rows); counterfactual_results.to_csv(RESULTS_DIR/"counterfactual_results.csv",index=False); display(counterfactual_results)
if not counterfactual_results.empty:
    means=counterfactual_results[["p_sdm_ihm","p_sdm_pseudo","p_sdm_real"]].mean(); fig,ax=plt.subplots(figsize=(7,4.5)); ax.bar(["IHM","Pseudo-SDM","Real SDM"],means)
    ax.set(ylim=(0,1),ylabel="Mean P(SDM)",title="Held-out counterfactual recording-condition probability"); plt.tight_layout(); fig.savefig(FIGURE_DIR/"counterfactual_probability.png",dpi=200); plt.show()
fig,ax=plt.subplots(figsize=(9,4)); ax.plot(f,20*np.log10(gain+1e-10)); ax.axhline(0,color="black",lw=.8); ax.set(xlabel="Frequency (Hz)",ylabel="Estimated amplitude gain (dB)",title="Training-only aggregate IHM→SDM spectral response")
plt.tight_layout(); fig.savefig(FIGURE_DIR/"estimated_channel_response.png",dpi=200); plt.show()


## 18. Clustered uncertainty

Utterances from the same speaker or meeting are not independent observations. Cluster bootstrap intervals therefore better reflect uncertainty over speakers or recording sessions.

Item-level bootstrap remains a secondary diagnostic. For recording-condition predictions, the notebook reports item, speaker-cluster, and meeting-cluster intervals where the required cluster labels are available. Each output row names its bootstrap unit explicitly.


In [ ]:
def bootstrap_ci(y_true, y_pred, metric, n=N_BOOTSTRAPS, seed=SEED):
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    generator, values = np.random.default_rng(seed), []
    for _ in range(n):
        indices = generator.integers(0, len(y_true), len(y_true))
        try:
            values.append(metric(y_true[indices], y_pred[indices]))
        except Exception:
            pass
    return (np.quantile(values, 0.025), np.quantile(values, 0.975)) if values else (np.nan, np.nan)

def cluster_bootstrap_ci(y_true, y_pred, clusters, metric, n=N_BOOTSTRAPS, seed=SEED):
    y_true, y_pred, clusters = np.asarray(y_true), np.asarray(y_pred), np.asarray(clusters)
    if not (len(y_true) == len(y_pred) == len(clusters)):
        raise ValueError("Truth, predictions, and cluster labels must have equal length")
    unique_clusters = np.unique(clusters)
    if len(unique_clusters) < 2:
        return np.nan, np.nan
    cluster_indices = {cluster: np.flatnonzero(clusters == cluster) for cluster in unique_clusters}
    generator, values = np.random.default_rng(seed), []
    for _ in range(n):
        sampled_clusters = generator.choice(unique_clusters, size=len(unique_clusters), replace=True)
        indices = np.concatenate([cluster_indices[cluster] for cluster in sampled_clusters])
        try:
            values.append(metric(y_true[indices], y_pred[indices]))
        except Exception:
            pass
    return (np.quantile(values, 0.025), np.quantile(values, 0.975)) if values else (np.nan, np.nan)

ci_rows = []
for name, (truth, prediction) in prediction_registry.items():
    is_condition_metric = "recording_condition" in name
    metric = (
        (lambda actual, predicted: balanced_accuracy_score(actual, predicted))
        if is_condition_metric
        else (lambda actual, predicted: f1_score(actual, predicted, average="macro"))
    )
    point = metric(truth, prediction)
    low, high = bootstrap_ci(truth, prediction, metric)
    ci_rows.append({
        "protocol_version": PROTOCOL_VERSION,
        "metric": name,
        "bootstrap_unit": "item",
        "point": point,
        "ci_95_low": low,
        "ci_95_high": high,
        "iterations": N_BOOTSTRAPS,
    })
    if name in prediction_clusters:
        for unit in ("speaker", "meeting"):
            low, high = cluster_bootstrap_ci(
                truth, prediction, prediction_clusters[name][unit], metric, seed=SEED
            )
            ci_rows.append({
                "protocol_version": PROTOCOL_VERSION,
                "metric": name,
                "bootstrap_unit": unit,
                "point": point,
                "ci_95_low": low,
                "ci_95_high": high,
                "iterations": N_BOOTSTRAPS,
            })

uncertainty = pd.DataFrame(ci_rows)
uncertainty.to_csv(RESULTS_DIR / "bootstrap_intervals.csv", index=False)
display(uncertainty)


## 19. Final comparison dashboard

The primary representation table contains only points produced by the standardized frozen logistic-probe protocol. Pair distance and neural-head performance remain separate diagnostics. CORAL is appended with an explicit target-aware scope and is not plotted as assumption-equivalent to source-only methods.

Missing values mean that a quantity was not defined or the required task was unavailable; values are never filled with guesses.


In [ ]:
final_summary = standardized_probe_results.copy()
if not final_summary.empty:
    final_summary["comparison_scope"] = "standardized_frozen_probe"
    final_summary["paired_embedding_distance"] = np.nan
    if not alignment_results.empty:
        distance_map = {
            f"PairAlign λ={row.lambda_pair:g}": row.matched_pair_distance
            for row in alignment_results.itertuples()
        }
        final_summary["paired_embedding_distance"] = final_summary.representation.map(distance_map)

if not coral_row.empty:
    coral_summary = pd.DataFrame([{
        "protocol_version": PROTOCOL_VERSION,
        "representation": "CORAL (UDA)",
        "representation_family": "target_aware_unlabeled",
        "mic_probe_accuracy": np.nan,
        "speaker_same_channel_f1": np.nan,
        "speaker_cross_channel_f1": coral_row.iloc[0].speaker_cross_channel_f1,
        "comparison_scope": "target_aware_adaptation_separate",
        "paired_embedding_distance": np.nan,
    }])
    final_summary = pd.concat([final_summary, coral_summary], ignore_index=True, sort=False)

final_summary.to_csv(RESULTS_DIR / "final_summary.csv", index=False)
display(final_summary)

baseline_export = mic_results.copy()
for task, table in (("speaker_closed_set_information", speaker_results), ("small_lexical_probe", lexical_results)):
    if not table.empty:
        addition = table.copy()
        addition.insert(1, "experiment", task)
        addition.insert(2, "model", "Logistic Regression")
        addition.insert(3, "representation", "Raw")
        baseline_export = pd.concat([baseline_export, addition], ignore_index=True, sort=False)
baseline_export.to_csv(RESULTS_DIR / "baseline_results.csv", index=False)


## 20. Automatically generated findings and hypothesis checks

The generated narrative keeps four concepts separate:

1. **Channel recoverability:** how accurately recording condition can be predicted.
2. **Cross-channel robustness:** how much downstream performance changes between same- and cross-condition tests.
3. **Geometric pair alignment:** how close matched IHM/SDM embeddings become.
4. **Representational channel leakage:** how well a frozen probe still recovers recording condition.

Pair-distance reduction is never treated as evidence that condition information was removed. Thresholds are declared before assessment, and H4a and H4b are reported separately.


In [ ]:
STRONG_RECOVERABILITY = 0.70
MODERATE_RECOVERABILITY = 0.58
MATERIAL_TASK_CHANGE = 0.05
MATERIAL_PAIR_REDUCTION_PERCENT = 5.0
MATERIAL_MIC_REDUCTION_POINTS = 0.05

raw_row = source_only_results[source_only_results.representation == "Raw"].iloc[0]
lines = [
    "### Computed findings",
    f"1. **Channel recoverability.** Raw features yielded {raw_row.mic_accuracy:.1%} balanced accuracy in the speaker-held-out recording-condition probe.",
]

if np.isfinite(raw_row.speaker_same_channel_f1) and np.isfinite(raw_row.speaker_cross_channel_f1):
    speaker_drop = raw_row.speaker_same_channel_f1 - raw_row.speaker_cross_channel_f1
    lines.append(
        f"2. **Cross-condition robustness.** Closed-set speaker-information macro-F1 changed from "
        f"{raw_row.speaker_same_channel_f1:.3f} same-condition to {raw_row.speaker_cross_channel_f1:.3f} "
        f"cross-condition (degradation {speaker_drop:+.3f})."
    )
else:
    speaker_drop = np.nan
    lines.append("2. **Cross-condition robustness.** The closed-set speaker-information probe was unavailable.")

source_candidate = best_norm
task_change = (
    source_candidate.speaker_cross_channel_f1 - raw_row.speaker_cross_channel_f1
    if np.isfinite(source_candidate.speaker_cross_channel_f1)
    and np.isfinite(raw_row.speaker_cross_channel_f1)
    else np.nan
)
normalization_text = (
    f"3. **Source-only preprocessing.** {source_candidate.representation} had the lowest validation-selected "
    f"recording-condition accuracy among source-only methods; test accuracy was {source_candidate.mic_accuracy:.1%}, "
    f"a change of {source_candidate.mic_accuracy - raw_row.mic_accuracy:+.1%} from Raw."
)
if np.isfinite(task_change):
    normalization_text += f" Cross-condition speaker-probe F1 changed by {task_change:+.3f}."
lines.append(normalization_text)

if not alignment_results.empty:
    selected = alignment_results.loc[alignment_results.lambda_pair == best_lambda].iloc[0]
    lines.append(
        f"4. **Geometric pair alignment.** Validation selected λ={best_lambda:g}; matched test-pair cosine "
        f"distance changed by {selected.pair_distance_reduction_percent:+.1f}% relative to λ=0, while frozen "
        f"cross-condition speaker-probe F1 changed by {selected.task_f1_change:+.3f}."
    )
    lines.append(
        f"5. **Representational channel leakage.** At the selected λ, the frozen recording-condition probe "
        f"scored {selected.mic_probe_accuracy:.1%}, a reduction of "
        f"{100 * selected.mic_probe_reduction_points:+.1f} percentage points relative to λ=0."
    )

if not counterfactual_results.empty:
    means = counterfactual_results[["p_sdm_ihm", "p_sdm_pseudo", "p_sdm_real"]].mean()
    lines.append(
        f"6. **Spectral diagnostic.** Mean P(SDM) moved from {means.p_sdm_ihm:.1%} on original IHM to "
        f"{means.p_sdm_pseudo:.1%} on pseudo-SDM; real SDM was {means.p_sdm_real:.1%}. "
        "This describes sensitivity to the estimated coloration without isolating a causal factor."
    )

h1 = (
    "supported" if raw_row.mic_accuracy >= STRONG_RECOVERABILITY
    else "partially supported" if raw_row.mic_accuracy >= MODERATE_RECOVERABILITY
    else "not supported"
)
h2 = (
    "not evaluated" if not np.isfinite(speaker_drop)
    else "supported" if speaker_drop >= MATERIAL_TASK_CHANGE
    else "partially supported" if speaker_drop > 0
    else "not supported"
)
leakage_reduction = raw_row.mic_accuracy - source_candidate.mic_accuracy
h3 = (
    "supported" if leakage_reduction >= MATERIAL_MIC_REDUCTION_POINTS
    and (not np.isfinite(task_change) or task_change >= -MATERIAL_TASK_CHANGE)
    else "partially supported" if leakage_reduction > 0
    else "not supported"
)

if alignment_results.empty:
    h4a = h4b = "not evaluated"
else:
    selected = alignment_results.loc[alignment_results.lambda_pair == best_lambda].iloc[0]
    preserves_task = selected.task_f1_change >= -MATERIAL_TASK_CHANGE
    h4a = (
        "supported" if selected.pair_distance_reduction_percent >= MATERIAL_PAIR_REDUCTION_PERCENT and preserves_task
        else "partially supported" if selected.pair_distance_reduction_percent > 0
        else "not supported"
    )
    h4b = (
        "supported" if selected.mic_probe_reduction_points >= MATERIAL_MIC_REDUCTION_POINTS
        else "partially supported" if selected.mic_probe_reduction_points > 0
        else "not supported"
    )

lines += [
    "",
    "### Pre-declared hypothesis assessment",
    f"- **H1:** {h1}",
    f"- **H2:** {h2}",
    f"- **H3:** {h3}",
    f"- **H4a — geometric alignment with task preservation:** {h4a}",
    f"- **H4b — reduced recoverable recording-condition information:** {h4b}",
]
display(Markdown("\n\n".join(lines)))


## 21. Limitations

- IHM and SDM differ in recording condition, including distance, room acoustics, reverberation, noise, placement, gain behavior, and hardware. The contrast does not isolate microphone hardware.
- AMI meeting speech does not represent every language, speaking style, room, device, or population.
- Balanced round-robin sampling improves diversity but does not turn speakers or meetings into a representative population sample.
- The speaker-information task is closed-set: identities are shared across partitions while utterance pairs are disjoint.
- The small lexical probe covers only frequent single-word classes supported by the sampled training data.
- Speaker- and meeting-cluster bootstrap intervals are more appropriate than item-only intervals, but few clusters can still produce unstable uncertainty estimates.
- MFCC-CMN and MFCC-CMVN affect only cepstral features; the combined representation retains non-cepstral cues.
- CORAL uses unlabeled target-condition training data and answers a target-aware adaptation question.
- Pair alignment reduces a geometric distance by construction; only a separate frozen probe tests whether recording-condition information remains recoverable.
- The estimated spectral response is linear and aggregate. A distant-microphone recording is not merely a filtered headset recording.
- Engineered features are interpretable and lightweight but are not modern foundation-model representations.

MicCheck is a controlled probe of recording-condition sensitivity and shortcut information, not proof of causal microphone bias.

## 22. Future research

Useful next steps include a full v2 rerun, more independent meetings and rooms, a second paired-condition corpus, speaker verification on unseen identities, nested grouped model selection, and preregistered external validation.


## 23. Export v2 results

Generated outputs include protocol-version metadata, the sampled pair manifest, dataset audit, feature and amplitude ablations, grouped recording-condition evaluations, standardized frozen-probe comparisons, clustered uncertainty, alignment diagnostics, counterfactual diagnostics, and figures.

The repository's committed MicCheck_results directory remains the immutable v1 results snapshot. Running this notebook writes fresh v2 artifacts under results and creates a separate archive; it does not revise historical CSV values in place.


In [ ]:
archive_base = PROJECT_DIR / f"MicCheck_results_protocol_v{PROTOCOL_VERSION.replace('.', '_')}"
archive_path = Path(shutil.make_archive(str(archive_base), "zip", root_dir=RESULTS_DIR))
print("Saved v2 results:")
for path in sorted(RESULTS_DIR.rglob("*")):
    if path.is_file():
        print(" -", path.relative_to(RESULTS_DIR))
print(f"\nArchive: {archive_path} ({archive_path.stat().st_size / 2**20:.1f} MiB)")


The generated protocol-v2 archive is saved in the current working directory. In Google Colab it can be downloaded from the Files panel; no Google Drive mount or repository clone is required.
